# Projet Nexovia : code complet

Maroua Nabih, Mathis Raffin, Axel Chasteauneuf.

Le projet a été réalisé en groupe avec une assistance IA pour certains développements, les commentaires, les expériences et la documentation. La répartition des contributions figure dans le README et les rapports.

Ce notebook contient tout le code Python du projet, dans le même ordre que le rapport :

1. le socle commun (lecture des données, vérification d'un planning, fonction objectif) ;
2. l'heuristique gloutonne qui construit la solution de départ S₀ ;
3. les deux recherches locales (première amélioration et plus forte pente) ;
4. les trois métaheuristiques : recuit simulé, recherche tabou (avec sa variante hybride ALNS) et algorithme génétique ;
5. l'approche multiobjectif : méthode ε-contrainte et recherche locale de Pareto (PLS) ;
6. les programmes qui lancent les campagnes d'expériences et refont les tableaux et figures ;
7. les compléments ajoutés après la soutenance.

**Utilisation.** Le notebook doit être placé à côté du dossier `donnees/` (les neuf instances N0 à N200) et du fichier `S0_communs.json` (les solutions de départ communes). Les cellules peuvent être exécutées dans l’ordre. Les campagnes longues sont désactivées par défaut avec `LANCER_CAMPAGNES = False` ; les exemples courts restent actifs. Le fichier `code/` contient aussi les modules prêts à importer, sans avoir à exécuter le notebook.

Les parties 1 à 3 et la recherche tabou sont écrites directement dans les cellules. Pour les autres méthodes, chaque cellule qui commence par `%%writefile` enregistre un fichier Python dans le dossier `code/` ; les petites cellules d'exemple qui suivent importent ces fichiers et lancent la méthode sur une instance. Le guide d'utilisation (dépendances, paramètres) est dans le complément au rapport.

Cette édition GitHub corrige l’ordre de deux cellules et harmonise les S0 des comparaisons finales. Les résultats historiques restent conservés séparément. Voir `docs/VALIDATION.md`.


In [ ]:
# dossier où les cellules %%writefile enregistrent les fichiers Python
import os, sys
os.makedirs("code", exist_ok=True)
sys.path.insert(0, "code")
# Activer explicitement pour les campagnes de plusieurs heures.
LANCER_CAMPAGNES = False


## 1. Socle commun

Ces fonctions sont utilisées par toutes les méthodes : elles lisent une instance, vérifient qu'un planning respecte toutes les contraintes et calculent son coût f = D + c_rep · R (distance totale plus coût d'un report fois le nombre de reports).

### 1.1 Lecture des données

Chaque instance est un dossier de six fichiers CSV (techniciens, interventions, disponibilités, lieux, trajets, paramètres). `lire_instance` les charge dans des tableaux pandas. On lit ensuite la petite instance N0 et on affiche ses tableaux pour vérifier la lecture.

In [ ]:
import pandas as pd
from pathlib import Path

def lire_instance(nom):
    # nom = chemin du dossier de l'instance (ex : "donnees/N50")
    # chaque instance est un dossier avec 6 fichiers csv séparés par des ;
    dossier = Path(nom)

    instance = {
        "techniciens": pd.read_csv(dossier / "techniciens.csv", sep=";"),
        "interventions": pd.read_csv(dossier / "interventions.csv", sep=";"),
        "disponibilites": pd.read_csv(dossier / "disponibilites.csv", sep=";"),
        "lieux": pd.read_csv(dossier / "lieux.csv", sep=";"),
        "trajets": pd.read_csv(dossier / "trajets.csv", sep=";"),
        "parametres": pd.read_csv(dossier / "parametres.csv", sep=";")
    }

    return instance

In [ ]:
# lecture de la petite instance N0 (8 interventions, 3 techniciens)
N0 = lire_instance("donnees/N0")

In [ ]:
# on affiche les tableaux pour vérifier que la lecture s'est bien passée
print("Techniciens :")
display(N0["techniciens"])

print("Interventions :")
display(N0["interventions"])

print("Disponibilités :")
display(N0["disponibilites"])

print("Paramètres :")
display(N0["parametres"])

In [ ]:
display(N0["trajets"].head())

### 1.2 Préparation des données

Filtrer un tableau pandas est lent, et les méthodes vérifient des millions de tournées. `preparer_instance` recopie donc une fois pour toutes les données dans des dictionnaires : distance et temps pour chaque couple de lieux, fenêtre horaire, durée et compétence de chaque intervention, domicile, compétences et créneaux de chaque technicien, et coût d'un report.

In [ ]:
def preparer_instance(N):
    # On recopie les tableaux pandas dans des dictionnaires.
    # Chercher une valeur dans un dictionnaire est beaucoup plus rapide que filtrer
    # un DataFrame, et ces fonctions sont appelées des millions de fois.

    # Distances et temps de trajet, avec la clé (origine, destination)
    N["dist"] = {}
    N["temps"] = {}

    for _, ligne in N["trajets"].iterrows():
        origine = ligne["origine"]
        destination = ligne["destination"]
        N["dist"][(origine, destination)] = ligne["distance_km"]
        N["temps"][(origine, destination)] = ligne["temps_min"]

    # Informations sur les interventions : durée, fenêtre horaire et compétence
    N["duree"] = {}
    N["debut"] = {}
    N["fin"] = {}
    N["comp"] = {}

    for _, ligne in N["interventions"].iterrows():
        i = ligne["intervention"]
        N["duree"][i] = ligne["duree_min"]
        N["debut"][i] = ligne["debut_fenetre_min"]
        N["fin"][i] = ligne["fin_fenetre_min"]
        N["comp"][i] = ligne["competence_requise"]

    # Informations sur les techniciens : domicile, compétences (1 ou 0)
    # et liste de leurs créneaux de disponibilité
    N["domicile"] = {}
    N["habilite"] = {}
    N["dispos"] = {}

    for _, ligne in N["techniciens"].iterrows():
        t = ligne["technicien"]

        N["domicile"][t] = ligne["domicile"]
        N["habilite"][t] = {
            "ELEC": ligne["ELEC"],
            "EAU": ligne["EAU"],
            "COMPTEUR": ligne["COMPTEUR"]
        }
        N["dispos"][t] = []

    # on trie les créneaux par heure de début pour les parcourir dans l'ordre
    lignes = N["disponibilites"].sort_values("debut_min")
    for _, ligne in lignes.iterrows():
        N["dispos"][ligne["technicien"]].append(
            (ligne["debut_min"], ligne["fin_min"])
        )

    # Liste des techniciens et coût d'un report (2e ligne de parametres.csv)
    N["techs"] = N["techniciens"]["technicien"].tolist()
    N["penalite"] = float(N["parametres"]["valeur"].iloc[1])

In [ ]:
# petit test : on vérifie quelques valeurs après la préparation
preparer_instance(N0)

print("Distance H1 → I1 :", N0["dist"][("H1", "I1")], "km")
print("Temps H1 → I1 :", N0["temps"][("H1", "I1")], "min")
print("Disponibilités A2 :", N0["dispos"]["A2"])
print("Pénalité :", N0["penalite"])

### 1.3 Vérification d'un planning et fonction objectif

- `verifier_missions` : chaque intervention apparaît exactement une fois (dans une tournée ou dans les reportées) ;
- `verifier_competences` : chaque technicien a la compétence de ses interventions ;
- `faire_trajet` : calcule l'heure d'arrivée d'un trajet, qui ne peut se faire que pendant les créneaux du technicien (un trajet peut être coupé par une pause) ;
- `verifier_tournee` : vérifie une seule tournée (compétences, créneaux, fenêtres horaires, retour au domicile). C'est la fonction la plus utilisée : les méthodes ne revérifient que les tournées qu'elles viennent de modifier ;
- `distance_tournee` et `f` : distance d'une tournée et coût total d'un planning ;
- `est_realisable` : vérification complète, utilisée pour contrôler les solutions finales.

In [ ]:
def verifier_missions(S, N):
    # Chaque intervention doit apparaître exactement une fois :
    # soit dans une tournée, soit dans les reportées.

    missions = []

    for tech in N["techniciens"]["technicien"]:
        missions += S[tech]

    missions += S["reportees"]

    toutes = N["interventions"]["intervention"].tolist()

    # si on trie les deux listes, elles doivent être identiques
    return sorted(missions) == sorted(toutes)

In [ ]:
def verifier_competences(S, N):
    # Un technicien ne peut faire que les interventions de ses compétences.
    # (version lente avec pandas, utilisée seulement dans est_realisable)

    for tech in N["techniciens"]["technicien"]:

        ligne_tech = N["techniciens"][
            N["techniciens"]["technicien"] == tech
        ].iloc[0]

        for mission in S[tech]:

            intervention = N["interventions"][
                N["interventions"]["intervention"] == mission
            ].iloc[0]

            competence = intervention["competence_requise"]

            if ligne_tech[competence] != 1:
                return False

    return True

In [ ]:
def lire_disponibilites(N, tech):
    # Renvoie les créneaux (debut, fin) d'un technicien, triés.
    # Sert à afficher_planning ; les autres fonctions utilisent N["dispos"].

    lignes = N["disponibilites"][
        N["disponibilites"]["technicien"] == tech
    ].sort_values("debut_min")

    disponibilites = []

    for _, ligne in lignes.iterrows():
        disponibilites.append(
            (ligne["debut_min"], ligne["fin_min"])
        )

    return disponibilites

In [ ]:
# petite tolérance pour éviter les erreurs d'arrondi sur les minutes
EPS = 1e-9

def faire_trajet(N, origine, destination, heure, disponibilites):
    # Fait rouler le technicien de origine à destination à partir de "heure".
    # Il ne peut rouler que pendant ses créneaux : s'il y a une pause, le trajet
    # est coupé en deux et reprend au créneau suivant.
    # Renvoie l'heure d'arrivée, ou None si le trajet ne tient pas dans la journée.

    if origine == destination:
        return heure

    reste = N["temps"][(origine, destination)]   # minutes de route qu'il reste à faire

    for debut, fin in disponibilites:

        # si on est avant le créneau, on attend son début
        heure = max(heure, debut)

        # créneau déjà fini : on passe au suivant
        if heure >= fin:
            continue

        temps_disponible = fin - heure

        # le reste du trajet tient dans ce créneau : on arrive
        if reste <= temps_disponible + EPS:
            return heure + reste

        # sinon on roule jusqu'à la fin du créneau et on continue au suivant
        reste = reste - temps_disponible
        heure = fin

    return None


def verifier_tournee(S, N, tech):
    # Vérifie la tournée d'un seul technicien : compétences, créneaux,
    # fenêtres horaires des interventions et retour au domicile.
    # C'est la fonction la plus utilisée : les métaheuristiques ne revérifient
    # que les tournées qu'elles viennent de modifier.

    domicile = N["domicile"][tech]
    disponibilites = N["dispos"][tech]


    # Compétences
    for mission in S[tech]:
        competence = N["comp"][mission]
        if N["habilite"][tech][competence] != 1:
            return False

    # départ du domicile au début du premier créneau
    heure = disponibilites[0][0]
    lieu = domicile

    for mission in S[tech]:

        # Aller à l'intervention
        heure = faire_trajet(N, lieu, mission, heure, disponibilites)

        if heure is None:
            return False

        # Chercher une heure possible pour la mission : elle doit tenir en entier
        # dans un créneau du technicien et finir avant la fin de sa fenêtre
        possible = False

        for debut, fin in disponibilites:

            debut_mission = max(heure, debut, N["debut"][mission])
            fin_mission = debut_mission + N["duree"][mission]

            if fin_mission <= fin + EPS and fin_mission <= N["fin"][mission] + EPS:
                heure = fin_mission
                possible = True
                break

        if possible == False:
            return False


        lieu = mission

    # Retour au domicile, lui aussi pendant les créneaux
    retour = faire_trajet(N, lieu, domicile, heure, disponibilites)

    return retour is not None


def distance_tournee(N, tech, missions):
    # Distance d'une tournée : domicile -> missions dans l'ordre -> domicile

    if len(missions) == 0:
        return 0

    domicile = N["domicile"][tech]
    tournee = [domicile] + missions + [domicile]

    distance = 0

    for i in range(len(tournee) - 1):
        distance += N["dist"][(tournee[i], tournee[i + 1])]

    return distance


def f(S, N):
    # Fonction objectif : distance totale + coût de report x nombre de reports

    distance_totale = 0

    for tech in N["techs"]:
        distance_totale += distance_tournee(N, tech, S[tech])


    return distance_totale + N["penalite"] * len(S["reportees"])

In [ ]:
def est_realisable(S, N):
    # Vérification complète d'une solution (plus lente) : on s'en sert
    # pour contrôler les solutions finales.

    if verifier_missions(S, N) == False:
        return False

    if verifier_competences(S, N) == False:
        return False

    for tech in N["techniciens"]["technicien"]:

        if verifier_tournee(S, N, tech) == False:
            return False

    return True

### 1.4 Test sur une solution construite à la main

On vérifie les fonctions sur une solution de N0 faite à la main : elle doit être réalisable, et on calcule la distance de chaque tournée puis son coût.

In [ ]:
# solution construite à la main au premier cours, pour tester les fonctions
S0 = {
    "A1": ["I1", "I4", "I2"],
    "A2": ["I3", "I6"],
    "A3": ["I5", "I8"],
    "reportees": ["I7"]
}

print("Solution réalisable :", est_realisable(S0, N0))

In [ ]:
# distance de chaque tournée de cette solution
print("Distance A1 :", distance_tournee(N0, "A1", S0["A1"]))
print("Distance A2 :", distance_tournee(N0, "A2", S0["A2"]))
print("Distance A3 :", distance_tournee(N0, "A3", S0["A3"]))

In [ ]:
# coût total = distance + pénalité x nombre de reports
print("valeur de s0 pour N0 :", round(f(S0, N0), 2))

## 2. Heuristique gloutonne

L'heuristique construit la solution de départ S₀. Les interventions sont prises par fin de fenêtre croissante (les plus urgentes d'abord). Chacune est insérée chez le technicien et à la position qui ajoutent le moins de kilomètres, en ne vérifiant que la tournée modifiée. Si aucune insertion n'est possible, l'intervention est reportée.

In [ ]:
def heuristique_gloutonne(N):
    # Construit la solution de départ S0.
    # On prend les interventions par fin de fenêtre croissante et on place chacune
    # là où elle ajoute le moins de kilomètres. Si elle ne rentre nulle part,
    # elle est reportée.

    # Au début, les tournées sont vides
    S = {}

    for tech in N["techs"]:
        S[tech] = []

    S["reportees"] = []

    # On commence par les interventions qui finissent le plus tôt
    interventions = N["interventions"].sort_values("fin_fenetre_min")

    for mission in interventions["intervention"]:

        meilleur_tech = None
        meilleure_position = None
        meilleur_cout = float("inf")

        # On essaie chaque technicien
        for tech in N["techs"]:

            ancienne_distance = distance_tournee(N, tech, S[tech])

            # On teste toutes les positions possibles
            for position in range(len(S[tech]) + 1):

                S[tech].insert(position, mission)

                # On vérifie seulement la tournée modifiée
                if verifier_tournee(S, N, tech):

                    nouvelle_distance = distance_tournee(N, tech, S[tech])
                    cout_ajoute = nouvelle_distance - ancienne_distance

                    if cout_ajoute < meilleur_cout:
                        meilleur_cout = cout_ajoute
                        meilleur_tech = tech
                        meilleure_position = position

                # On enlève la mission pour tester une autre position
                S[tech].pop(position)

        # On garde la meilleure insertion trouvée
        if meilleur_tech is not None:
            S[meilleur_tech].insert(meilleure_position, mission)

        else:
            S["reportees"].append(mission)

    return S

### Écriture du socle commun

Le glouton est maintenant défini : le socle complet peut être exporté pour les scripts Python.


In [ ]:
# On récupère le code des fonctions du socle définies plus haut (inspect.getsource)
# et on l'écrit dans code/socle.py, avec les imports et la constante EPS dont elles ont besoin.
import inspect

fonctions_socle = [lire_instance, preparer_instance, verifier_missions, verifier_competences,
                   lire_disponibilites, faire_trajet, verifier_tournee, distance_tournee,
                   f, est_realisable, heuristique_gloutonne]

with open("code/socle.py", "w", encoding="utf-8") as fichier:
    fichier.write("import pandas as pd\nfrom pathlib import Path\n\nEPS = 1e-9\n\n\n")
    for fonction in fonctions_socle:
        fichier.write(inspect.getsource(fonction) + "\n\n")

print("code/socle.py écrit :", len(fonctions_socle), "fonctions")

In [ ]:
# solution de départ obtenue par le glouton sur N0
S0 = heuristique_gloutonne(N0)

print("Solution initiale :")

for tech in N0["techniciens"]["technicien"]:
    print(tech, ":", S0[tech])

print("Interventions reportées :", S0["reportees"])

print("Solution réalisable :", est_realisable(S0, N0))

print("Coût initial :", round(f(S0, N0), 2))

In [ ]:
# glouton sur N10
N10 = lire_instance("donnees/N10")
preparer_instance(N10)

S10 = heuristique_gloutonne(N10)

print("Solution N10 :", S10)
print("Réalisable :", est_realisable(S10, N10))
print("Coût initial :", round(f(S10, N10), 2))

In [ ]:
# le coût d'un report est la 2e valeur de parametres.csv
print("Pénalité N0 :", N0["parametres"]["valeur"].iloc[1])
print("Pénalité N10 :", N10["parametres"]["valeur"].iloc[1])

### 2.1 Toutes les instances

On lit et on prépare les neuf instances, puis on applique le glouton à chacune : coût, nombre de reports, faisabilité et temps de calcul sont rassemblés dans un tableau.

In [ ]:
noms = ["N0", "N10", "N20", "N30", "N50", "N80", "N100", "N150", "N200"]

# vérification : nombre d'interventions de chaque instance
for nom in noms:
    N = lire_instance("donnees/" + nom)
    print(nom, ":", len(N["interventions"]), "interventions")

In [ ]:
# on lit toutes les instances et on les garde dans un dictionnaire
noms = ["N20", "N30", "N50", "N80", "N100", "N150", "N200"]

instances = {}

for nom in noms:
    instances[nom] = lire_instance("donnees/" + nom)

In [ ]:
# préparation des dictionnaires (distances, fenêtres, compétences...)
for nom in noms:
    N = instances[nom]
    preparer_instance(N)

print("Toutes les instances sont préparées.")

In [ ]:
# contrôle que la préparation a bien été faite
for nom in noms:
    print(nom, ":", "techs" in instances[nom])

In [ ]:
# Glouton sur toutes les instances : coût, nombre de reports, temps de calcul
import time

solutions_gloutonnes = {}
resultats_glouton = []

for nom in noms:

    N = instances[nom]

    debut = time.perf_counter()
    S = heuristique_gloutonne(N)
    duree = time.perf_counter() - debut

    solutions_gloutonnes[nom] = S

    cout = round(f(S, N), 2)
    reports = len(S["reportees"])
    realisable = est_realisable(S, N)

    resultats_glouton.append({
        "Instance": nom,
        "Coût initial": cout,
        "Reports": reports,
        "Réalisable": realisable,
        "Temps (s)": round(duree, 2)
   })

    print(nom, ":", cout, "| Reports :", reports,
          "| Réalisable :", realisable, "| Temps :", round(duree, 2), "s")

In [ ]:
# tableau récapitulatif du glouton, enregistré en csv
import pandas as pd

tableau_glouton = pd.DataFrame(resultats_glouton)

display(tableau_glouton)

tableau_glouton.to_csv(
    "resultats_glouton.csv",
    index=False,
    sep=";"
)

In [ ]:
# (on refait la préparation par sécurité)
for nom in instances:
    preparer_instance(instances[nom])

print("Toutes les instances sont préparées.")

In [ ]:
for nom in noms:
    print(nom, ":", "techs" in instances[nom])

In [ ]:
# on ajoute N0 et N10, préparées plus haut, au dictionnaire des instances
instances["N0"] = N0
instances["N10"] = N10

solutions_gloutonnes["N0"] = S0
solutions_gloutonnes["N10"] = S10
# Départs identiques pour les comparaisons finales ; distincts des gloutons exploratoires.
import json
with open("S0_communs.json", encoding="utf-8") as entree:
    solutions_communes = json.load(entree)


### 2.2 Affichage d'un planning

`afficher_planning` refait la journée de chaque technicien et affiche les heures de départ, de chaque trajet, de chaque intervention et du retour au domicile. Cela permet de vérifier une solution à la main.

In [ ]:
# convertit des minutes (ex : 495.5) en texte hh:mm:ss pour l'affichage
def afficher_heure(minutes):
    secondes = round(minutes * 60)
    h = secondes // 3600
    m = (secondes % 3600) // 60
    s = secondes % 60

    return f"{h:02d}:{m:02d}:{s:02d}"

In [ ]:
# Affiche le planning d'une solution réalisable : heure de départ, trajets,
# heure de chaque mission et retour au domicile, pour chaque technicien.
def afficher_planning(S, N):

    if not est_realisable(S, N):
        print("La solution n'est pas réalisable")
        return

    for tech in N["techniciens"]["technicien"]:

        print("\nTechnicien :", tech)

        ligne = N["techniciens"][
            N["techniciens"]["technicien"] == tech
        ].iloc[0]

        domicile = ligne["domicile"]
        disponibilites = lire_disponibilites(N, tech)

        heure = disponibilites[0][0]
        lieu = domicile

        if len(S[tech]) == 0:
            print("Aucune mission")
            continue

        print("Départ :", afficher_heure(heure))

        for mission in S[tech]:

            # Déplacement vers la mission
            depart = heure
            heure = faire_trajet(N, lieu, mission, heure, disponibilites)

            print("Trajet", lieu, "→", mission, ":",
                  afficher_heure(depart), "à", afficher_heure(heure))

            intervention = N["interventions"][
                N["interventions"]["intervention"] == mission
            ].iloc[0]

            # Heure de début et de fin de la mission
            for debut, fin in disponibilites:

                debut_mission = max(
                    heure, debut, intervention["debut_fenetre_min"]
                )

                fin_mission = debut_mission + intervention["duree_min"]

                if fin_mission <= fin + EPS and fin_mission <= intervention["fin_fenetre_min"] + EPS:
                    print("Mission", mission, ":",
                          afficher_heure(debut_mission), "à",
                          afficher_heure(fin_mission))
                    heure = fin_mission
                    break

            lieu = mission

        # Retour à la maison
        heure = faire_trajet(N, lieu, domicile, heure, disponibilites)

        print("Retour au domicile :", afficher_heure(heure))

    print("\nInterventions reportées :", S["reportees"])

In [ ]:
# planning de la solution gloutonne de N0
afficher_planning(S0, N0)

In [ ]:
afficher_planning(S10, N10)

## 3. Recherches locales

### 3.1 Voisinage relocate

Un voisin est obtenu en retirant une intervention de sa liste (une tournée ou les reportées) et en la remettant ailleurs : à une autre position, chez un autre technicien compétent, ou dans les reportées. `generer_voisins` donne les voisins réalisables un par un, sans les garder tous en mémoire, et ne revérifie que les deux tournées modifiées.

In [ ]:
# Voisinage relocate : on retire une mission de sa liste (tournée ou reportées)
# et on la remet ailleurs (autre position, autre technicien ou reportées).
# C'est un générateur (yield) : les voisins sont donnés un par un, sans tout
# garder en mémoire. Seuls les voisins réalisables sont renvoyés.
def generer_voisins(S, N):

    listes = N["techs"] + ["reportees"]

    # on parcourt chaque mission de chaque liste
    for source in listes:
        for position_source, mission in enumerate(S[source]):

            # On retire la mission de sa liste
            route_source = S[source][:position_source] + S[source][position_source + 1:]

            for destination in listes:

                if source == destination == "reportees":
                    continue

                # Vérifier la compétence avant de tester les positions
                if destination != "reportees":
                    competence = N["comp"][mission]

                    if N["habilite"][destination][competence] != 1:
                        continue

                if source == destination:
                    base = route_source
                else:
                    base = S[destination]

                if destination == "reportees":
                    positions = [len(base)]
                else:
                    positions = range(len(base) + 1)

                for position in positions:

                    # Éviter le même planning et les échanges en double
                    if source == destination:
                        if position == position_source or position == position_source + 1:
                            continue

                    nouvelle_route = base[:position] + [mission] + base[position:]

                    voisin = {cle: S[cle] for cle in S}

                    if source == destination:
                        voisin[source] = nouvelle_route
                    else:
                        voisin[source] = route_source
                        voisin[destination] = nouvelle_route

                    # Vérifier seulement les tournées modifiées
                    if source != "reportees":
                        if not verifier_tournee(voisin, N, source):
                            continue

                    if destination != "reportees" and destination != source:
                        if not verifier_tournee(voisin, N, destination):
                            continue

                    mouvement = {
                        "mission": mission,
                        "source": source,
                        "destination": destination,
                        "position_source": position_source,
                        "position_destination": position
                    }

                    # on renvoie le voisin et la description du mouvement
                    yield voisin, mouvement

In [ ]:
# même voisinage mais rangé dans deux listes (pratique pour compter ou afficher)
def voisinage(S, N):

    voisins = []
    mouvements = []

    for V, mouvement in generer_voisins(S, N):
        voisins.append(V)
        mouvements.append(mouvement)

    return voisins, mouvements

In [ ]:
# test du voisinage sur N0 : nombre de voisins et les 3 premiers
voisins, mouvements = voisinage(S0, N0)

print("Nombre de voisins réalisables :", len(voisins))

for i in range(min(3, len(voisins))):
    print("\nVoisin", i + 1)
    print("Mouvement :", mouvements[i])
    print("Solution :", voisins[i])
    print("Coût :", round(f(voisins[i], N0), 2))

In [ ]:
# contrôle : tous les voisins renvoyés doivent être réalisables
for S, N in [(S0, N0), (S10, N10)]:

    voisins, mouvements = voisinage(S, N)

    print("Nombre de voisins :", len(voisins))
    print("Tous réalisables :", all(est_realisable(V, N) for V in voisins))

### 3.2 Descentes

- **Première amélioration** : on parcourt les voisins et on se déplace vers le premier qui fait baisser le coût ;
- **Plus forte pente** : on examine tous les voisins et on se déplace vers le meilleur.

Les deux s'arrêtent dans un optimum local, quand aucun voisin n'est meilleur. On les compare sur N0 et N10.

In [ ]:
# Descente « première amélioration » : on parcourt les voisins et on prend
# le premier qui fait baisser le coût. On recommence jusqu'à ce qu'aucun voisin
# ne soit meilleur (optimum local).
def premiere_amelioration(S0, N):

    S = {cle: S0[cle][:] for cle in S0}

    while True:

        amelioration = False
        cout_actuel = f(S, N)

        for V, mouvement in generer_voisins(S, N):

            if f(V, N) < cout_actuel - 1e-9:
                S = V
                amelioration = True
                break

        if amelioration == False:
            break

    return S

In [ ]:
# Descente « plus forte pente » : à chaque étape on regarde TOUS les voisins
# et on prend le meilleur. Plus lent par étape, mais chaque pas est le plus grand possible.
def plus_forte_pente(S0, N):

    S = {cle: S0[cle][:] for cle in S0}

    while True:

        meilleur = S
        meilleur_cout = f(S, N)

        # On examine tous les voisins
        for V, mouvement in generer_voisins(S, N):

            cout_voisin = f(V, N)

            if cout_voisin < meilleur_cout - 1e-9:
                meilleur = V
                meilleur_cout = cout_voisin

        # On s'arrête si aucun voisin n'est meilleur
        if meilleur == S:
            break

        S = meilleur

    return S

In [ ]:
# comparaison des deux descentes sur N0
S_premiere = premiere_amelioration(S0, N0)
S_pente = plus_forte_pente(S0, N0)

print("Coût initial :", round(f(S0, N0), 2))
print("Première amélioration :", round(f(S_premiere, N0), 2))
print("Plus forte pente :", round(f(S_pente, N0), 2))

print("Réalisable première amélioration :", est_realisable(S_premiere, N0))
print("Réalisable plus forte pente :", est_realisable(S_pente, N0))

In [ ]:
# comparaison des deux descentes sur N10
S_premiere10 = premiere_amelioration(S10, N10)
S_pente10 = plus_forte_pente(S10, N10)

print("Coût initial :", round(f(S10, N10), 2))
print("Première amélioration :", round(f(S_premiere10, N10), 2))
print("Plus forte pente :", round(f(S_pente10, N10), 2))

print("Réalisable première amélioration :", est_realisable(S_premiere10, N10))
print("Réalisable plus forte pente :", est_realisable(S_pente10, N10))

In [ ]:
# on regarde si les deux descentes arrivent à la même solution
print("Première amélioration :")
for cle, missions in S_premiere10.items():
    print(cle, ":", missions)

print("\nPlus forte pente :")
for cle, missions in S_pente10.items():
    print(cle, ":", missions)

print("\nSolutions identiques :", S_premiere10 == S_pente10)

### 3.3 Campagne finale sur les mêmes départs

Les deux descentes sont évaluées depuis `S0_communs.json`. Les gloutons recalculés plus haut sont conservés comme exemples et départs exploratoires. Cette campagne est facultative et désactivée par défaut.


In [ ]:
if LANCER_CAMPAGNES:
    # première amélioration sur toutes les instances, depuis la solution gloutonne
    import time

    noms = ["N0", "N10", "N20", "N30", "N50", "N80", "N100", "N150", "N200"]

    solutions_premiere = {}

    for nom in noms:

        N = instances[nom]
        S = solutions_communes[nom]

        # Première amélioration
        debut = time.perf_counter()
        S_finale = premiere_amelioration(S, N)
        temps = time.perf_counter() - debut

        solutions_premiere[nom] = S_finale

        print(nom)
        print("Coût final :", round(f(S_finale, N), 2))
        print("Reports :", len(S_finale["reportees"]))
        print("Temps :", round(temps, 2), "secondes")
        print("Réalisable :", est_realisable(S_finale, N))
else:
    print("Campagne longue désactivée ; activer LANCER_CAMPAGNES pour la lancer.")


In [ ]:
if LANCER_CAMPAGNES:
    # plus forte pente sur toutes les instances
    import time

    solutions_pente = {}
    resultats_pente = []

    for nom in noms:

        N = instances[nom]
        S = solutions_communes[nom]

        # On applique la plus forte pente
        debut = time.perf_counter()
        S_finale = plus_forte_pente(S, N)
        temps = time.perf_counter() - debut

        solutions_pente[nom] = S_finale

        cout = round(f(S_finale, N), 2)
        reports = len(S_finale["reportees"])
        realisable = est_realisable(S_finale, N)

        resultats_pente.append({
            "Instance": nom,
            "Coût final": cout,
            "Reports": reports,
            "Réalisable": realisable,
            "Temps (s)": round(temps, 2)
        })

        print(nom, ":", cout, "| Reports :", reports,
              "| Réalisable :", realisable,
              "| Temps :", round(temps, 2), "s", flush=True)
else:
    print("Campagne longue désactivée ; activer LANCER_CAMPAGNES pour la lancer.")


In [ ]:
if LANCER_CAMPAGNES:
    # Gains de la comparaison finale : même référence pour toutes les méthodes.
    for nom in noms:
        N = instances[nom]
        cout_initial = f(solutions_communes[nom], N)
        cout_premiere = f(solutions_premiere[nom], N)
        cout_pente = f(solutions_pente[nom], N)
        print(nom, "| S0 commun :", round(cout_initial, 2),
              "| Première :", round(100 * (cout_initial - cout_premiere) / cout_initial, 2), "%",
              "| Pente :", round(100 * (cout_initial - cout_pente) / cout_initial, 2), "%")

else:
    print("Campagne longue désactivée ; activer LANCER_CAMPAGNES pour la lancer.")


## 4. Métaheuristiques

### 4.1 Recuit simulé

Le fichier `recuit.py` contient trois fonctions :

- `voisin_aleatoire` : au lieu de construire tout le voisinage, on tire au hasard les quatre éléments d'un mouvement relocate (liste de départ, intervention, destination, position) et on ne vérifie que les deux tournées modifiées ;
- `calculer_T0` : la température initiale est choisie pour qu'une dégradation moyenne autour de S₀ soit acceptée avec la probabilité τ₀, soit T₀ = -Δ⁺ / ln τ₀ ;
- `recuit_simule` : à chaque itération, un voisin est accepté s'il améliore le coût, sinon avec la probabilité exp(−Δ/T). La température est multipliée par α tous les k · n itérations ; quand elle passe sous T₀/1000, on repart de la meilleure solution avec T = T₀ (réchauffage). L'arrêt se fait au bout du temps imparti. Le coût d'un voisin est calculé en ne recalculant que la distance des tournées modifiées.

Réglage retenu : τ₀ = 0,1, α = 0,99, k = 20, 180 secondes.

In [ ]:
%%writefile code/recuit.py
# Recuit simulé.
# On utilise seulement les fonctions du socle commun (verifier_tournee,
# distance_tournee, f, est_realisable) et le même mouvement que le voisinage
# relocate (on enlève une mission et on la remet ailleurs).

import math
import random
import time

from socle import *


# ---------------------------------------------------------------------------
# 1. Tirer un voisin au hasard
# ---------------------------------------------------------------------------
# generer_voisins construit TOUS les voisins relocate. Le recuit n'en utilise
# qu'un par itération, donc on tire directement les 4 choix du mouvement au
# hasard (liste de départ, mission, destination, position) et on ne construit
# qu'un seul voisin. C'est ce qui nous a fait passer d'environ 2 à 30 000
# itérations par seconde sur N200.

def voisin_aleatoire(S, N):

    listes = N["techs"] + ["reportees"]

    # Choisir une liste non vide, puis une mission dans cette liste
    source = random.choice([l for l in listes if len(S[l]) > 0])
    position_source = random.randrange(len(S[source]))
    mission = S[source][position_source]

    # Choisir une destination : un technicien compétent ou les reportées
    competence = N["comp"][mission]
    destinations = [t for t in N["techs"] if N["habilite"][t][competence] == 1] + ["reportees"]
    destination = random.choice(destinations)

    # reportée -> reportée : ça ne change rien, on rejette le tirage
    if source == "reportees" and destination == "reportees":
        return None, None

    # On retire la mission de sa liste
    route_source = S[source][:position_source] + S[source][position_source + 1:]

    if source == destination:
        base = route_source
    else:
        base = S[destination]

    # Choisir une position (pour les reportées, l'ordre ne compte pas : à la fin)
    if destination == "reportees":
        position = len(base)
    else:
        position = random.randint(0, len(base))

    # Remettre la mission à sa place ne change rien
    if source == destination and position == position_source:
        return None, None

    nouvelle_route = base[:position] + [mission] + base[position:]

    # Construction du voisin (on copie les listes pour ne jamais modifier S)
    voisin = {cle: S[cle][:] for cle in S}
    if source == destination:
        voisin[source] = nouvelle_route
    else:
        voisin[source] = route_source
        voisin[destination] = nouvelle_route

    # Seules les 2 tournées modifiées peuvent être devenues impossibles,
    # donc on ne vérifie qu'elles
    for tech in [source, destination]:
        if tech != "reportees" and not verifier_tournee(voisin, N, tech):
            return None, None

    mouvement = {"mission": mission, "source": source, "destination": destination,
                 "position_source": position_source, "position_destination": position}
    return voisin, mouvement


# ---------------------------------------------------------------------------
# 2. Température initiale T0
# ---------------------------------------------------------------------------
# On tire des voisins de S0 et on garde ceux qui dégradent le coût.
# Avec leur dégradation moyenne d, on choisit T0 pour qu'au début une
# dégradation moyenne soit acceptée avec la probabilité tau0 :
#     exp(-d / T0) = tau0   donc   T0 = -d / ln(tau0)
# Comme ça T0 s'adapte tout seul à l'échelle des coûts de chaque instance.

def calculer_T0(S0, N, tau0, nb=500):

    cout_S0 = f(S0, N)
    degradations = []
    essais = 0

    # au plus 50 x 500 = 25 000 tirages (en pratique moins de 9 000 suffisent)
    while len(degradations) < nb and essais < 50 * nb:
        essais += 1
        V, mouvement = voisin_aleatoire(S0, N)
        if V is not None:
            delta = f(V, N) - cout_S0
            if delta > 0:
                degradations.append(delta)

    # cas limite : aucune dégradation trouvée, on prend une valeur par défaut
    if len(degradations) == 0:
        return 1.0

    moyenne = sum(degradations) / len(degradations)
    return -moyenne / math.log(tau0)


# ---------------------------------------------------------------------------
# 3. Recuit simulé
# ---------------------------------------------------------------------------
# Paramètres :
#   tau0      : taux d'acceptation au départ (sert à calculer T0)
#   alpha     : refroidissement, T = alpha * T à la fin de chaque palier
#   k_palier  : un palier dure k_palier * n itérations (n = nombre d'interventions)
#   temps_max : budget en secondes, c'est le seul critère d'arrêt
# Réchauffage : quand T passe sous T0 / 1000 et qu'il reste du temps, on repart
# de la meilleure solution avec T = T0.
#
# Pour aller vite on ne recalcule que la distance des 2 tournées modifiées :
#   nouvelle distance = distance actuelle + (distance des tournées modifiées après - avant)
#
# Options utilisées par les autres fonctions de ce fichier :
#   cout        : autre fonction à minimiser (pour l'epsilon-contrainte)
#   duree_cycle : refroidissement calé sur le temps, en secondes (epsilon-contrainte et essais P)
#   Q           : variante "P essais" du cours. L'essai s'arrête après Q itérations sans
#                 améliorer la meilleure solution, et il n'y a plus de réchauffage.
#                 Avec Q = None (par défaut), le recuit est celui du rapport.

def recuit_simule(S0, N, temps_max=60, graine=None, tau0=0.1, alpha=0.99, k_palier=20,
                  T0=None, cout=None, duree_cycle=None, garder_trace=False, Q=None):

    random.seed(graine)
    debut = time.perf_counter()

    # par défaut on minimise f = D + pénalité x R
    if cout is None:
        def cout(D, R):
            return D + N["penalite"] * R

    if T0 is None:
        T0 = calculer_T0(S0, N, tau0)
    T_min = T0 / 1000

    # longueur d'un palier (au moins 10 itérations pour les toutes petites instances)
    n = len(N["interventions"])
    if duree_cycle is None:
        palier = max(10, k_palier * n)
    else:
        palier = max(10, n)

    # Solution courante : on garde la distance de chaque tournée pour le calcul rapide
    S = {cle: S0[cle][:] for cle in S0}
    distance = {tech: distance_tournee(N, tech, S[tech]) for tech in N["techs"]}
    D = sum(distance.values())
    R = len(S["reportees"])
    cout_actuel = cout(D, R)

    meilleure = {cle: S[cle][:] for cle in S}
    meilleur_cout = cout_actuel
    temps_meilleur = 0.0
    historique = [(0.0, meilleur_cout)]          # (temps, meilleur coût) à chaque amélioration
    trace = []                                   # (temps, coût courant, température)

    T = T0
    debut_cycle = time.perf_counter()
    iterations = acceptes = rechauffages = 0
    dans_palier = 0
    sans_amelioration = 0                        # c'est le compteur J du cours

    while time.perf_counter() - debut < temps_max:

        iterations += 1
        sans_amelioration += 1
        if Q is not None and sans_amelioration > Q:
            break                                # fin de l'essai (critère Q du cours)
        V, mouvement = voisin_aleatoire(S, N)

        # V = None : voisin impossible, l'itération est perdue et on retire au hasard
        if V is not None:

            # Variation de distance : seulement les tournées modifiées
            nouvelles = {}
            for tech in [mouvement["source"], mouvement["destination"]]:
                if tech != "reportees":
                    nouvelles[tech] = distance_tournee(N, tech, V[tech])
            D_voisin = D + sum(nouvelles[t] - distance[t] for t in nouvelles)
            R_voisin = len(V["reportees"])
            cout_voisin = cout(D_voisin, R_voisin)

            difference = cout_voisin - cout_actuel

            # Règle de Metropolis : on accepte toujours si c'est mieux,
            # sinon avec la probabilité exp(-difference / T)
            if difference <= 0 or random.random() < math.exp(-difference / T):
                S = V
                distance.update(nouvelles)
                D, R, cout_actuel = D_voisin, R_voisin, cout_voisin
                acceptes += 1

                # nouvelle meilleure solution (1e-9 pour ignorer les égalités à l'arrondi près)
                if cout_actuel < meilleur_cout - 1e-9:
                    meilleure = {cle: S[cle][:] for cle in S}
                    meilleur_cout = cout_actuel
                    temps_meilleur = time.perf_counter() - debut
                    historique.append((temps_meilleur, meilleur_cout))
                    sans_amelioration = 0

        # Fin de palier : on refroidit
        dans_palier += 1
        if dans_palier >= palier:
            dans_palier = 0

            if garder_trace:
                trace.append((time.perf_counter() - debut, cout_actuel, T))

            if duree_cycle is None:
                # refroidissement classique
                T = alpha * T
            else:
                # refroidissement calé sur le temps : T va de T0 à T0/1000
                # exactement pendant duree_cycle secondes
                t_cycle = time.perf_counter() - debut_cycle
                T = T0 * (1 / 1000) ** min(1.0, t_cycle / duree_cycle)
                if t_cycle >= duree_cycle:
                    T = T_min / 2

            # Réchauffage depuis la meilleure solution (sauf en mode "essais du cours")
            if T < T_min and Q is None:
                T = T0
                rechauffages += 1
                debut_cycle = time.perf_counter()
                S = {cle: meilleure[cle][:] for cle in meilleure}
                distance = {tech: distance_tournee(N, tech, S[tech]) for tech in N["techs"]}
                D = sum(distance.values())
                R = len(S["reportees"])
                cout_actuel = meilleur_cout

    duree = time.perf_counter() - debut
    historique.append((duree, meilleur_cout))
    infos = {"iterations": iterations, "acceptes": acceptes, "rechauffages": rechauffages,
             "T0": T0, "palier": palier, "temps": duree, "temps_meilleur": temps_meilleur,
             "historique": historique, "trace": trace}
    return meilleure, meilleur_cout, infos

Exemple : le recuit sur N50.

In [ ]:
import json
import recuit

N50 = lire_instance("donnees/N50")
preparer_instance(N50)
S0_N50 = json.load(open("S0_communs.json"))["N50"]

S, cout, infos = recuit.recuit_simule(S0_N50, N50, temps_max=5, graine=1)
print("Coût de départ :", round(f(S0_N50, N50), 2), "| coût final :", round(cout, 2))
print("Itérations :", infos["iterations"], "| T0 :", round(infos["T0"], 2), "| réalisable :", est_realisable(S, N50))

### 4.2 Recherche tabou

À chaque itération, on parcourt le voisinage relocate et on garde 40 voisins non tabous tirés au hasard (tirage par réservoir). On se déplace vers le meilleur des 40, même s'il est moins bon que la solution actuelle. Le mouvement inverse (remettre l'intervention chez son ancien technicien) devient alors tabou pendant L itérations. Chaque appel réalise cinq essais de 180 secondes et relève le meilleur coût à 60, 120 et 180 secondes.

In [ ]:
# Recherche tabou. Pour chaque essai (5 graines), on part de S0 et pendant 180 s :
#  - on parcourt le voisinage relocate et on garde 40 candidats non tabous tirés au hasard
#    (tirage par réservoir, pour ne pas stocker tous les voisins) ;
#  - on prend le meilleur des 40, même s'il est moins bon que la solution actuelle ;
#  - le mouvement inverse (remettre la mission chez son ancien technicien) devient tabou
#    pendant L itérations.
# On relève le meilleur coût à 60, 120 et 180 s.
import time
import random
from collections import deque

def recherche_tabou(S0, N, L):

    resultats = []
    solutions = []

    for essai in range(1, 6):

        # générateur aléatoire propre à l'essai, pour pouvoir refaire le même calcul
        rng = random.Random(essai)

        S = {cle: S0[cle][:] for cle in S0}
        meilleure = {cle: S[cle][:] for cle in S}
        meilleur_cout = f(S, N)

        # deque(maxlen=L) : quand la liste est pleine, le plus ancien tabou sort tout seul
        liste_tabou = deque(maxlen=L)
        couts = {60: None, 120: None, 180: None}

        debut = time.perf_counter()

        while time.perf_counter() - debut < 180:

            candidats = []
            nb = 0

            # Générer des voisins et tirer 40 candidats au hasard
            for V, mouvement in generer_voisins(S, N):

                temps = time.perf_counter() - debut

                for t in couts:
                    if temps >= t and couts[t] is None:
                        couts[t] = meilleur_cout

                if temps >= 180:
                    break

                # un mouvement est tabou si on remet la mission là où elle était avant
                cle = (mouvement["mission"], mouvement["destination"])

                if cle in liste_tabou:
                    continue

                nb += 1

                # tirage par réservoir : chaque voisin autorisé a la même chance d'être parmi les 40
                if len(candidats) < 40:
                    candidats.append((V, mouvement))
                else:
                    j = rng.randrange(nb)
                    if j < 40:
                        candidats[j] = (V, mouvement)

            if time.perf_counter() - debut >= 180 or not candidats:
                break

            # Choisir le meilleur candidat autorisé
            rng.shuffle(candidats)
            S, mouvement = min(candidats, key=lambda x: f(x[0], N))

            if time.perf_counter() - debut >= 180:
                break

            # Mémoriser le mouvement inverse
            liste_tabou.append((
                mouvement["mission"],
                mouvement["source"]
            ))

            if f(S, N) < meilleur_cout:
                meilleur_cout = f(S, N)
                meilleure = {cle: S[cle][:] for cle in S}

        temps_reel = time.perf_counter() - debut

        # si l'essai s'est arrêté avant un seuil, on prend le meilleur coût final
        for t in couts:
            if couts[t] is None:
                couts[t] = meilleur_cout

        resultats.append({
            "Essai": essai,
            "Coût 60": round(couts[60], 2),
            "Coût 120": round(couts[120], 2),
            "Coût 180": round(couts[180], 2),
            "Reports": len(meilleure["reportees"]),
            "Temps": round(temps_reel, 2)
        })

        solutions.append(meilleure)

    return resultats, solutions

In [ ]:
if LANCER_CAMPAGNES:
    # premier test du tabou sur N0 avec L = 10
    N = instances["N0"]
    S0 = solutions_communes["N0"]

    resultats_N0, solutions_N0 = recherche_tabou(S0, N, L=10)

    display(pd.DataFrame(resultats_N0))

    print("Coût initial :", round(f(S0, N), 2))
    print("Meilleur coût :", min(r["Coût 180"] for r in resultats_N0))
    print("Toutes réalisables :",
          all(est_realisable(S, N) for S in solutions_N0))
else:
    print("Campagne longue désactivée ; activer LANCER_CAMPAGNES pour la lancer.")


In [ ]:
if LANCER_CAMPAGNES:
    # même test sur N10
    N = instances["N10"]
    S0 = solutions_communes["N10"]

    resultats_N10, solutions_N10 = recherche_tabou(S0, N, L=10)

    display(pd.DataFrame(resultats_N10))

    print("Coût initial :", round(f(S0, N), 2))
    print("Meilleur coût :", min(r["Coût 180"] for r in resultats_N10))
    print("Toutes réalisables :",
          all(est_realisable(S, N) for S in solutions_N10))
else:
    print("Campagne longue désactivée ; activer LANCER_CAMPAGNES pour la lancer.")


**Réglage de la longueur L de la liste tabou.** Les valeurs 10, 20 et 30 sont testées sur six instances d'apprentissage ; la valeur qui donne le meilleur gain moyen à 180 secondes est ensuite évaluée sur les trois autres instances.

In [ ]:
if LANCER_CAMPAGNES:
    # Sensibilité de la longueur L de la liste tabou (10, 20, 30) sur les instances
    # d'apprentissage. On sauvegarde après chaque instance pour ne rien perdre si le calcul est interrompu.
    train = ["N0", "N10", "N30", "N80", "N100", "N200"]

    sensibilite = []
    solutions_train = {}

    for L in [10, 20, 30]:

        for nom in train:

            N = instances[nom]
            S0 = solutions_communes[nom]
            cout_initial = f(S0, N)

            # Réutiliser les résultats déjà calculés
            if L == 10 and nom == "N0":
                resultats, solutions = resultats_N0, solutions_N0

            elif L == 10 and nom == "N10":
                resultats, solutions = resultats_N10, solutions_N10

            else:
                resultats, solutions = recherche_tabou(S0, N, L)

            solutions_train[(nom, L)] = solutions

            for r in resultats:

                sensibilite.append({
                    "Instance": nom,
                    "L": L,
                    "Essai": r["Essai"],
                    "Coût 60": r["Coût 60"],
                    "Coût 120": r["Coût 120"],
                    "Coût 180": r["Coût 180"],
                    "Gain 60": 100 * (cout_initial - r["Coût 60"]) / cout_initial,
                    "Gain 120": 100 * (cout_initial - r["Coût 120"]) / cout_initial,
                    "Gain 180": 100 * (cout_initial - r["Coût 180"]) / cout_initial,
                    "Reports": r["Reports"],
                    "Temps": r["Temps"]
                })

            print(nom, "| L =", L, "terminé", flush=True)

            # Sauvegarde après chaque instance
            pd.DataFrame(sensibilite).to_csv(
                "sensibilite_tabou.csv", index=False
            )
else:
    print("Campagne longue désactivée ; activer LANCER_CAMPAGNES pour la lancer.")


In [ ]:
if LANCER_CAMPAGNES:
    # gain moyen par valeur de L : on retient celle qui donne le meilleur gain à 180 s
    tableau = pd.DataFrame(sensibilite)

    comparaison = tableau.groupby("L")[
        ["Gain 60", "Gain 120", "Gain 180"]
    ].mean().round(2)

    display(comparaison)

    L_choisi = int(comparaison["Gain 180"].idxmax())

    print("Valeur de L retenue :", L_choisi)
else:
    print("Campagne longue désactivée ; activer LANCER_CAMPAGNES pour la lancer.")


In [ ]:
if LANCER_CAMPAGNES:
    # test de la valeur retenue sur les instances qui n'ont pas servi au réglage
    test = ["N20", "N50", "N150"]

    resultats_test = []
    solutions_test = {}

    for nom in test:

        N = instances[nom]
        S0 = solutions_communes[nom]

        resultats, solutions = recherche_tabou(S0, N, L_choisi)

        solutions_test[nom] = solutions

        for r in resultats:

            resultats_test.append({
                "Instance": nom,
                "Essai": r["Essai"],
                "Coût 60": r["Coût 60"],
                "Coût 120": r["Coût 120"],
                "Coût 180": r["Coût 180"],
                "Reports": r["Reports"],
                "Temps": r["Temps"]
            })

        print(nom, "| Réalisable :",
              all(est_realisable(S, N) for S in solutions),
              flush=True)

        pd.DataFrame(resultats_test).to_csv(
            "test_tabou.csv", index=False
        )

    display(pd.DataFrame(resultats_test))
else:
    print("Campagne longue désactivée ; activer LANCER_CAMPAGNES pour la lancer.")


In [ ]:
if LANCER_CAMPAGNES:
    # résumé final du tabou (instances d'apprentissage + de test)
    resultats_train = tableau[tableau["L"] == L_choisi]

    resultats_finaux = pd.concat([
        resultats_train,
        pd.DataFrame(resultats_test)
    ])

    resume = resultats_finaux.groupby("Instance").agg(
        Meilleur_cout=("Coût 180", "min"),
        Cout_moyen=("Coût 180", "mean"),
        Ecart_type=("Coût 180", "std"),
        Temps_moyen=("Temps", "mean")
    ).round(2)

    display(resume)

    resume.to_csv("resume_tabou.csv")
else:
    print("Campagne longue désactivée ; activer LANCER_CAMPAGNES pour la lancer.")


**Contrôle du blocage.** Ce test montre si tous les mouvements deviennent tabous. L’arrêt anticipé sur les petites instances est une limite connue de cette version, sans aspiration ; ce contrôle ne prétend pas supprimer le blocage.


In [ ]:
# Vérifie que la liste tabou ne bloque pas la recherche : sur 50 itérations,
# on compte les voisins et ceux qui restent autorisés. Si aucun n'est autorisé, c'est bloqué.
import random
from collections import deque

def verifier_blocage(nom, L=30):

    N = instances[nom]
    S = {cle: solutions_gloutonnes[nom][cle][:] for cle in solutions_gloutonnes[nom]}

    liste_tabou = deque(maxlen=L)
    rng = random.Random(1)

    for iteration in range(1, 51):

        candidats = []
        total = 0
        autorises = 0

        for V, mouvement in generer_voisins(S, N):

            total += 1

            cle = (mouvement["mission"], mouvement["destination"])

            if cle in liste_tabou:
                continue

            autorises += 1

            if len(candidats) < 40:
                candidats.append((V, mouvement))
            else:
                j = rng.randrange(autorises)
                if j < 40:
                    candidats[j] = (V, mouvement)

        print("Itération :", iteration,
              "| Voisins :", total,
              "| Autorisés :", autorises)

        if not candidats:
            print("ARRÊT : aucun voisin autorisé.")
            return

        rng.shuffle(candidats)
        S, mouvement = min(candidats, key=lambda x: f(x[0], N))

        liste_tabou.append((
            mouvement["mission"],
            mouvement["source"]
        ))

    print("50 itérations terminées sans blocage.")

In [ ]:
# test du blocage sur les deux petites instances, avec la plus longue liste (L = 30)
print("=== N0 ===")
verifier_blocage("N0", L=30)

print("\n=== N10 ===")
verifier_blocage("N10", L=30)

### 4.3 Variante hybride ALNS + tabou

**Amelioration par IA**

À chaque itération, une partie de la solution est détruite (10 à 25 % des interventions sont reportées, au hasard ou parmi les plus coûteuses), puis réparée (insertion gloutonne ou par regret), avant quelques pas de recherche tabou. Les quatre couples destruction / réparation ont un poids qui augmente lorsqu'ils donnent de bons résultats, et une solution moins bonne peut être acceptée pour explorer une autre région.

In [ ]:
# Méthode hybride ALNS + tabou (amélioration proposée avec l'aide d'une IA, voir la déclaration).
# À chaque itération : on détruit une partie de la solution (on reporte 10 à 25 % des missions),
# on la répare (insertion gloutonne ou par regret), puis on fait quelques pas de tabou.
# Les 4 couples destruction / réparation ont un poids qui augmente quand ils marchent bien.
import time
import random
import math

def copier_solution(S):
    return {cle: valeur[:] for cle, valeur in S.items()}


# Étape 1 : retirer plusieurs interventions
def detruire_hybride(S, N, rng, methode, quantite):

    S = copier_solution(S)

    for _ in range(quantite):

        missions = [
            (t, i, m)
            for t in N["techs"]
            for i, m in enumerate(S[t])
        ]

        if not missions:
            break

        if methode == "aleatoire":
            t, i, m = rng.choice(missions)

        else:
            # Retirer une intervention coûteuse en distance
            def cout_retrait(x):
                t, i, m = x
                sans = S[t][:i] + S[t][i+1:]
                return (distance_tournee(N, t, S[t])
                        - distance_tournee(N, t, sans))

            t, i, m = max(missions, key=cout_retrait)

        S[t].pop(i)
        S["reportees"].append(m)

    return S


# Étape 2 : reconstruire les tournées
def reparer_hybride(S, N, methode, debut, temps_max):

    S = copier_solution(S)

    while time.perf_counter() - debut < temps_max:

        choix = None

        for mission in S["reportees"]:

            options = []

            for t in N["techs"]:

                if time.perf_counter() - debut >= temps_max:
                    return S

                if N["habilite"][t][N["comp"][mission]] != 1:
                    continue

                d0 = distance_tournee(N, t, S[t])

                for pos in range(len(S[t]) + 1):

                    S[t].insert(pos, mission)

                    if verifier_tournee(S, N, t):

                        surplus = distance_tournee(N, t, S[t]) - d0

                        if surplus < N["penalite"] - 1e-9:
                            options.append((surplus, t, pos))

                    S[t].pop(pos)

            if not options:
                continue

            options.sort()
            d, t, pos = options[0]

            if methode == "gloutonne":
                score = N["penalite"] - d

            else:
                # regret = écart entre la 2e meilleure insertion et la meilleure :
                # on insère d'abord les missions qu'on risque de ne plus pouvoir bien placer
                # Réparation par regret
                if len(options) > 1:
                    second = min(N["penalite"], options[1][0])
                else:
                    second = N["penalite"]

                score = second - d

            if choix is None or score > choix[0]:
                choix = (score, mission, t, pos)

        if choix is None:
            break

        _, mission, t, pos = choix

        S["reportees"].remove(mission)
        S[t].insert(pos, mission)

    return S


# Étape 3 : méthode hybride ALNS + Tabou
def hybride_alns_tabou(S0, N, L, temps_max=180, graine=1):

    rng = random.Random(graine)
    debut = time.perf_counter()

    S = copier_solution(S0)
    meilleure = copier_solution(S0)
    meilleur_cout = f(S0, N)

    evolution = [(0.0, meilleur_cout)]

    # Quatre combinaisons destruction / réparation
    operateurs = [
        (a, b)
        for a in ("aleatoire", "couteux")
        for b in ("gloutonne", "regret")
    ]

    # au départ tous les opérateurs ont le même poids
    poids = {op: 1.0 for op in operateurs}
    iterations = 0

    while time.perf_counter() - debut < temps_max:

        # Choisir un opérateur selon ses performances
        op = rng.choices(
            operateurs,
            weights=[poids[o] for o in operateurs]
        )[0]

        planifiees = sum(len(S[t]) for t in N["techs"])

        # nombre de missions retirées : entre 10 et 25 % des missions planifiées
        q = max(1, round(
            planifiees * rng.uniform(0.10, 0.25)
        ))

        candidat = detruire_hybride(S, N, rng, op[0], q)

        candidat = reparer_hybride(
            candidat, N, op[1], debut, temps_max
        )

        # Amélioration locale par recherche tabou
        liste_tabou = []
        amelioration = False

        for _ in range(L + 5):

            if time.perf_counter() - debut >= temps_max:
                break

            candidats = []
            nb = 0

            for V, m in generer_voisins(candidat, N):

                if time.perf_counter() - debut >= temps_max:
                    break

                if (m["mission"], m["destination"]) in liste_tabou:
                    continue

                nb += 1

                if len(candidats) < 40:
                    candidats.append((V, m))
                else:
                    j = rng.randrange(nb)
                    if j < 40:
                        candidats[j] = (V, m)

            if not candidats:
                break

            rng.shuffle(candidats)

            candidat, mouvement = min(
                candidats,
                key=lambda x: f(x[0], N)
            )

            liste_tabou.append((
                mouvement["mission"],
                mouvement["source"]
            ))

            if len(liste_tabou) > L:
                liste_tabou.pop(0)

            cout = f(candidat, N)

            if cout < meilleur_cout - 1e-9:
                meilleure = copier_solution(candidat)
                meilleur_cout = cout
                amelioration = True

                evolution.append((
                    time.perf_counter() - debut,
                    meilleur_cout
                ))

        # Accepter parfois une solution moins bonne
        # pour explorer une autre région
        cout_candidat = f(candidat, N)
        cout_actuel = f(S, N)

        # température qui baisse avec le temps (comme dans un recuit)
        temperature = max(
            0.1,
            0.1 * f(S0, N) *
            (1 - (time.perf_counter() - debut) / temps_max)
        )

        accepte = (
            cout_candidat <= cout_actuel
            or rng.random() < math.exp(
                min(0, (cout_actuel - cout_candidat) / temperature)
            )
        )

        if accepte:
            S = candidat

        # Adapter le poids des opérateurs
        # récompense : 6 si nouveau record, 2 si accepté, 1 sinon
        recompense = 6 if amelioration else 2 if accepte else 1

        poids[op] = 0.9 * poids[op] + 0.1 * recompense

        iterations += 1

    # Coûts aux paliers de temps, si atteints
    couts = {
        t: min(c for moment, c in evolution if moment <= t)
        for t in (60, 120, 180)
        if t <= temps_max
    }

    infos = {
        "cout": round(meilleur_cout, 2),
        "temps": round(time.perf_counter() - debut, 2),
        "iterations": iterations,
        "couts": couts
    }

    return meilleure, infos

In [ ]:
# Les mêmes fonctions sont disponibles dans un module pour les scripts et les tests.
import inspect
from pathlib import Path
fonctions_locales = [generer_voisins, voisinage, premiere_amelioration, plus_forte_pente,
                    recherche_tabou, copier_solution, detruire_hybride, reparer_hybride, hybride_alns_tabou]
prefixe_local = "from socle import *\nimport time, random, math\nfrom collections import deque\n\n"
Path("code/local_tabou.py").write_text(prefixe_local + "\n\n".join(inspect.getsource(f) for f in fonctions_locales), encoding="utf-8")


In [ ]:
# essai rapide de l'hybride sur N10 (10 s)
N = instances["N10"]
S0 = solutions_communes["N10"]

S_hybride, infos = hybride_alns_tabou(
    S0, N,
    L=30,
    temps_max=10,
    graine=1
)

print("Coût glouton :", round(f(S0, N), 2))
print("Coût hybride :", round(f(S_hybride, N), 2))
print("Nombre de reports :", len(S_hybride["reportees"]))
print("Réalisable :", est_realisable(S_hybride, N))
print("Temps :", infos["temps"], "secondes")
print("Itérations :", infos["iterations"])

In [ ]:
if LANCER_CAMPAGNES:
    # 5 essais de 10 s sur N10
    import pandas as pd

    N = instances["N10"]
    S0 = solutions_communes["N10"]

    resultats_hybride = []

    for essai in range(1, 6):

        S, infos = hybride_alns_tabou(
            S0, N,
            L=30,
            temps_max=10,
            graine=essai
        )

        resultats_hybride.append({
            "Essai": essai,
            "Coût": round(f(S, N), 2),
            "Reports": len(S["reportees"]),
            "Temps": infos["temps"],
            "Réalisable": est_realisable(S, N)
        })

    display(pd.DataFrame(resultats_hybride))
else:
    print("Campagne longue désactivée ; activer LANCER_CAMPAGNES pour la lancer.")


In [ ]:
if LANCER_CAMPAGNES:
    # essais rapides de 10 s sur les instances de test
    import pandas as pd

    resultats_hybride_test = []

    for nom in ["N20", "N50", "N150"]:

        N = instances[nom]
        S0 = solutions_communes[nom]

        for essai in range(1, 6):

            S, infos = hybride_alns_tabou(
                S0, N,
                L=30,
                temps_max=10,
                graine=essai
            )

            resultats_hybride_test.append({
                "Instance": nom,
                "Essai": essai,
                "Coût": round(f(S, N), 2),
                "Reports": len(S["reportees"]),
                "Temps": infos["temps"],
                "Réalisable": est_realisable(S, N)
            })

        print(nom, "terminé", flush=True)

        pd.DataFrame(resultats_hybride_test).to_csv(
            "hybride_test_rapide.csv", index=False
        )

    display(pd.DataFrame(resultats_hybride_test))
else:
    print("Campagne longue désactivée ; activer LANCER_CAMPAGNES pour la lancer.")


In [ ]:
if LANCER_CAMPAGNES:
    # un essai complet de 180 s sur les instances de test
    for nom in ["N20", "N50", "N150"]:

        N = instances[nom]
        S0 = solutions_communes[nom]

        S, infos = hybride_alns_tabou(
            S0, N,
            L=30,
            temps_max=180,
            graine=1
        )

        print(nom)
        print("Coût hybride :", round(f(S, N), 2))
        print("Reports :", len(S["reportees"]))
        print("Réalisable :", est_realisable(S, N))
        print("Temps :", infos["temps"], "s")
        print("--------------------", flush=True)
else:
    print("Campagne longue désactivée ; activer LANCER_CAMPAGNES pour la lancer.")


### 4.4 Algorithme génétique

Un individu est un planning complet. Le fichier `genetique.py` contient :

- `individu_aleatoire_ag` : un nouvel individu, construit comme le glouton mais avec un ordre des interventions mélangé et des insertions choisies parmi les trois meilleures ;
- `croisement_ag` : l'enfant hérite, pour chaque technicien, de la tournée de l'un des deux parents, sans copier deux fois une intervention ; les interventions restantes sont ensuite réinsérées (réparation) ;
- `mutation_ag` : déplacement ou échange d'interventions, en gardant un planning réalisable ;
- `algorithme_genetique` : population de 20 individus, sélection par tournoi de 2, croisement avec la probabilité 0,8, mutation avec la probabilité 0,25, conservation des deux meilleurs et refus des doublons. Une partie de chaque génération est faite d'individus nouveaux (20 %, puis 40 % avec une mutation plus forte après dix générations sans amélioration). La diversité de la population est suivie au cours du temps.

In [ ]:
%%writefile code/genetique.py
# Partie Axel : algorithme génétique.
# Une solution (un individu) est le planning lui-même : une liste ordonnée de missions
# par technicien, plus la liste des reportées. On réutilise f, est_realisable,
# verifier_tournee et distance_tournee du socle commun.
# Les fonctions finissent par _ag pour ne pas se mélanger avec celles des autres parties.

from socle import *
import random
import time
import pandas as pd


def copie_ag(S):
    # copie d'un planning (on copie chaque liste pour ne pas modifier l'original)
    return {cle: missions[:] for cle, missions in S.items()}


def cle_ag(S, N):
    # "Empreinte" d'un planning, pour repérer les doublons dans la population.
    # L'ordre des reports n'a aucune importance, donc on les trie.
    return (tuple(tuple(S[t]) for t in N["techs"]),
            tuple(sorted(S["reportees"])))


def completer_ag(S, ordre, N, rng, limite, diversifier=False):
    # Essaie d'insérer les missions reportées (dans l'ordre donné) sans rendre le
    # planning impossible. Une mission reste reportée si aucune insertion n'est retenue.
    # limite = heure (perf_counter) à laquelle il faut s'arrêter.
    for mission in ordre:
        if time.perf_counter() >= limite:
            break
        if mission not in S["reportees"]:
            continue

        # on teste toutes les positions chez tous les techniciens
        choix = []
        for tech in N["techs"]:
            ancienne_distance = distance_tournee(N, tech, S[tech])
            for position in range(len(S[tech]) + 1):
                if time.perf_counter() >= limite:
                    return S
                S[tech].insert(position, mission)
                if verifier_tournee(S, N, tech):
                    ajout = distance_tournee(N, tech, S[tech]) - ancienne_distance
                    choix.append((ajout, tech, position))
                S[tech].pop(position)

        if choix:
            choix.sort()
            # Diversification : on choisit au hasard parmi les 3 meilleures insertions.
            # Sinon on prend la meilleure.
            ajout, tech, position = rng.choice(choix[:3]) if diversifier else choix[0]
            # sans diversification, on n'insère que si ça coûte moins cher que le report
            if diversifier or ajout <= N["penalite"]:
                S[tech].insert(position, mission)
                S["reportees"].remove(mission)
    return S


def individu_aleatoire_ag(N, rng, limite):
    # Nouvel individu : même principe que le glouton, mais avec un ordre des missions
    # mélangé et des insertions un peu au hasard, pour avoir des individus différents.
    ordre = list(N["duree"])
    rng.shuffle(ordre)
    S = {tech: [] for tech in N["techs"]}
    S["reportees"] = ordre[:]
    return completer_ag(S, ordre, N, rng, limite, diversifier=True)


def croisement_ag(parent1, parent2, N, rng, limite):
    # L'enfant part de "tout reporté" puis récupère des tournées des parents.
    enfant = {tech: [] for tech in N["techs"]}
    enfant["reportees"] = list(N["duree"])

    # Pour chaque technicien, hériter de la tournée d'un des deux parents.
    # Une mission déjà prise ailleurs n'est pas copiée une seconde fois.
    techniciens = N["techs"][:]
    rng.shuffle(techniciens)
    for tech in techniciens:
        parent = rng.choice([parent1, parent2])
        for mission in parent[tech]:
            if time.perf_counter() >= limite:
                return enfant
            if mission in enfant["reportees"]:
                enfant[tech].append(mission)
                # si la tournée devient impossible, on enlève la mission
                if verifier_tournee(enfant, N, tech):
                    enfant["reportees"].remove(mission)
                else:
                    enfant[tech].pop()

    # Réparation : on essaie de réinsérer les missions encore reportées.
    ordre = enfant["reportees"][:]
    rng.shuffle(ordre)
    return completer_ag(enfant, ordre, N, rng, limite)


def mutation_ag(S, N, rng, limite):
    # Petit changement aléatoire. On fait jusqu'à 20 tentatives et on garde la première
    # qui donne un planning différent et réalisable. Sinon on renvoie une copie de S.
    listes = N["techs"] + ["reportees"]
    for _ in range(20):
        if time.perf_counter() >= limite:
            break
        voisin = copie_ag(S)
        non_vides = [cle for cle in listes if voisin[cle]]
        if not non_vides:
            break
        source = rng.choice(non_vides)
        i = rng.randrange(len(voisin[source]))

        if rng.random() < 0.7:
            # 70 % du temps : déplacer une mission (la réordonner, la reporter ou la réinsérer)
            destination = rng.choice(listes)
            if source == destination == "reportees":
                continue
            mission = voisin[source].pop(i)
            position = rng.randrange(len(voisin[destination]) + 1)
            voisin[destination].insert(position, mission)
        else:
            # 30 % du temps : échanger deux missions ; avec les reports, ça fait un remplacement
            destination = rng.choice(non_vides)
            j = rng.randrange(len(voisin[destination]))
            voisin[source][i], voisin[destination][j] = (
                voisin[destination][j], voisin[source][i])

        if cle_ag(voisin, N) != cle_ag(S, N) and est_realisable(voisin, N):
            return voisin
    return copie_ag(S)


def algorithme_genetique(S0, N, taille_population=20, proba_croisement=0.8,
                        proba_mutation=0.25, budgets=(60, 120, 180),
                        graines=(1, 2, 3, 4, 5), diversite=True):
    # Lance une exécution par graine. On relève le meilleur coût à chaque budget
    # (60, 120, 180 s) dans la même exécution.
    # diversite=False ne sert qu'au test d'ablation (population sans immigrants ni refus des doublons).

    # quelques contrôles sur les paramètres
    if taille_population < 4:
        raise ValueError("La population doit contenir au moins 4 individus.")
    if not 0 <= proba_croisement <= 1 or not 0 <= proba_mutation <= 1:
        raise ValueError("Les probabilités doivent être entre 0 et 1.")
    budgets = sorted(set(budgets))
    if not budgets or budgets[0] <= 0:
        raise ValueError("Les budgets doivent être strictement positifs.")
    if not est_realisable(S0, N):
        raise ValueError("La solution initiale n'est pas réalisable.")

    resultats, solutions = [], []
    for graine in graines:
        rng = random.Random(graine)      # générateur propre à cette graine
        meilleure = copie_ag(S0)
        meilleur_cout = f(S0, N)
        couts = {budget: None for budget in budgets}
        historique = [(0.0, meilleur_cout)]
        generations = stagnation = 0
        suivi_population = []
        dernier_suivi = -1.0
        debut = time.perf_counter()
        limite = debut + budgets[-1]

        def observer(S):
            # Calcule le coût d'un nouvel individu et met à jour le meilleur connu.
            nonlocal meilleure, meilleur_cout
            cout = f(S, N)
            temps = time.perf_counter() - debut
            # Un résultat trouvé après un seuil ne compte pas avant ce seuil :
            # on fige la valeur d'un budget dès qu'on le dépasse.
            for budget in budgets:
                if temps >= budget and couts[budget] is None:
                    couts[budget] = meilleur_cout
            if temps <= budgets[-1] and cout < meilleur_cout - 1e-9:
                meilleure, meilleur_cout = copie_ag(S), cout
                historique.append((temps, cout))
            return cout

        # Population initiale : S0 + des individus aléatoires.
        # Chaque individu est stocké avec son coût.
        population = [(copie_ag(S0), meilleur_cout)]
        vus = {cle_ag(S0, N)}
        for _ in range(10 * taille_population):
            if len(population) >= taille_population or time.perf_counter() >= limite:
                break
            if diversite:
                S = individu_aleatoire_ag(N, rng, limite)
            else:
                # témoin sans diversité : simples mutations de S0
                S = copie_ag(S0)
                for _ in range(rng.randint(1, 3)):
                    S = mutation_ag(S, N, rng, limite)
            cout = observer(S)
            cle = cle_ag(S, N)
            if not diversite or cle not in vus:
                population.append((S, cout))
                vus.add(cle)

        def selection():
            # Tournoi de 2 : on tire deux individus et on garde le meilleur.
            # Ça garde une pression de sélection modérée.
            candidats = rng.sample(population, min(2, len(population)))
            return min(candidats, key=lambda individu: individu[1])[0]

        # Boucle des générations, jusqu'à la fin du budget
        while time.perf_counter() < limite:
            ancien_record = meilleur_cout
            population.sort(key=lambda individu: individu[1])
            nouvelle = population[:2]  # Élitisme : on garde les deux meilleurs.
            vus = {cle_ag(S, N) for S, cout in nouvelle}

            # Après 10 générations sans progrès, on renforce la diversification :
            # mutation à 0,7 au moins et 40 % d'immigrants au lieu de 20 %.
            mutation = max(proba_mutation, 0.7) if diversite and stagnation >= 10 else proba_mutation
            proportion = 0.4 if stagnation >= 10 else 0.2
            nb_immigrants = max(1, int(taille_population * proportion)) if diversite else 0

            for _ in range(10 * taille_population):
                if len(nouvelle) >= taille_population or time.perf_counter() >= limite:
                    break
                if len(nouvelle) < 2 + nb_immigrants:
                    # immigrant : individu tout neuf
                    enfant = individu_aleatoire_ag(N, rng, limite)
                else:
                    parent1, parent2 = selection(), selection()
                    if rng.random() < proba_croisement:
                        enfant = croisement_ag(parent1, parent2, N, rng, limite)
                    else:
                        enfant = copie_ag(parent1)
                    if rng.random() < mutation:
                        for _ in range(rng.randint(1, 3)):
                            enfant = mutation_ag(enfant, N, rng, limite)

                cout = observer(enfant)
                cle = cle_ag(enfant, N)
                if not diversite or cle not in vus:  # Refuser les doublons dans la population.
                    nouvelle.append((enfant, cout))
                    vus.add(cle)

            population = nouvelle
            generations += 1

            # Suivi de la population environ toutes les 0,25 s (pour les graphiques de diversité)
            temps_suivi = time.perf_counter() - debut
            if temps_suivi - dernier_suivi >= 0.25 and temps_suivi <= budgets[-1]:
                cout_population = sum(c for _, c in population) / len(population)
                uniques = len({cle_ag(s, N) for s, _ in population}) / len(population)

                # Diversité des affectations et des arcs des tournées, par rapport au meilleur membre.
                def arcs(solution):
                    # ensemble des trajets (technicien, de, vers) + des reports
                    ensemble = set()
                    for t in N["techs"]:
                        route = [N["domicile"][t]] + solution[t] + [N["domicile"][t]]
                        ensemble.update((t, a, b) for a, b in zip(route, route[1:]))
                    ensemble.update(("report", m) for m in solution["reportees"])
                    return ensemble
                reference = arcs(min(population, key=lambda x: x[1])[0])
                ecarts = []
                for solution, _ in population:
                    a = arcs(solution)
                    # distance de Jaccard : 1 - (arcs communs / arcs au total)
                    ecarts.append(1 - len(a & reference) / max(1, len(a | reference)))
                suivi_population.append({"temps": temps_suivi, "generation": generations,
                    "cout_moyen": cout_population, "cout_min": min(c for _, c in population),
                    "meilleur": meilleur_cout, "fraction_unique": uniques,
                    "diversite_arcs": sum(ecarts) / len(ecarts), "taille": len(population)})
                dernier_suivi = temps_suivi

            # compteur de stagnation : remis à 0 dès que le meilleur s'améliore
            stagnation = 0 if meilleur_cout < ancien_record - 1e-9 else stagnation + 1

        # fin de l'exécution : on complète les budgets pas encore atteints
        temps_reel = time.perf_counter() - debut
        for budget in budgets:
            if couts[budget] is None:
                couts[budget] = meilleur_cout
        assert est_realisable(meilleure, N)
        resultats.append({
            "Essai": graine,
            **{f"Coût {b}": couts[b] for b in budgets},
            "Reports": len(meilleure["reportees"]),
            "Temps": temps_reel,
            "Générations": generations,
            "Population finale": len(population),
            "Historique": historique,
            "Suivi population": suivi_population
        })
        solutions.append(meilleure)
    return resultats, solutions

Exemple : le génétique sur N50.

In [ ]:
import genetique

resultats, solutions = genetique.algorithme_genetique(S0_N50, N50, budgets=(5,), graines=(1,))
print("Coût final :", round(resultats[0]["Coût 5"], 2), "| générations :", resultats[0]["Générations"])

## 5. Approche multiobjectif

On minimise maintenant séparément la distance D et le nombre de reports R.

### 5.1 Méthode ε-contrainte

On résout une suite de problèmes « minimiser D sous R ≤ ε » avec le recuit simulé, pour 11 valeurs de ε entre 0 et n (toutes les valeurs si n ≤ 12) :

- la contrainte est remplacée par une pénalité D + M · max(0, R − ε), avec M = 100 c_rep ;
- les valeurs de ε sont traitées dans l'ordre croissant, chaque recuit partant de la solution du ε précédent ;
- les 180 secondes sont partagées entre les ε, et la température décroît selon le temps disponible pour faire un cycle complet ;
- on garde les points non dominés et on mesure la qualité du front par son hypervolume.

In [ ]:
%%writefile code/epsilon_contrainte.py
# Méthode epsilon-contrainte : on résout une suite de problèmes mono-objectif
# (min D sous R <= epsilon) avec le recuit de recuit.py.

import random

from recuit import *

# ---------------------------------------------------------------------------
# Méthode epsilon-contrainte
# ---------------------------------------------------------------------------
#   min D(S)   sous   R(S) <= epsilon
# La contrainte est remplacée par une pénalité : coût = D + M x max(0, R - epsilon)
# avec M = 100 x pénalité. Au début on avait mis un coût infini, mais inf - inf
# donne nan et le recuit n'acceptait plus aucun voisin.
# Les epsilon sont traités dans l'ordre croissant : chaque recuit part de la solution
# du epsilon précédent, qui respecte déjà R <= epsilon.
# Chaque epsilon n'a qu'environ 16 s, d'où le refroidissement calé sur le temps.

def objectifs(S, N):
    # les deux objectifs : distance D (arrondie au mètre) et nombre de reports R
    R = len(S["reportees"])
    D = f(S, N) - N["penalite"] * R
    return round(D, 3), R


def domine(a, b):
    # a domine b : aussi bon sur D et sur R, et strictement meilleur sur au moins un
    return a[0] <= b[0] + 1e-9 and a[1] <= b[1] and (a[0] < b[0] - 1e-9 or a[1] < b[1])


def filtrer_non_domines(points):
    # points = liste de (D, R, solution). On enlève les doublons puis les points dominés.
    uniques = {}
    for p in points:
        cle = (round(p[0], 3), p[1])
        if cle not in uniques:
            uniques[cle] = p
    pts = list(uniques.values())
    nd = [p for p in pts if not any(domine((q[0], q[1]), (p[0], p[1])) for q in pts if q is not p)]
    return sorted(nd, key=lambda p: (p[1], p[0]))


def hypervolume_2d(front, ref):
    # Aire dominée par le front et bornée par le point de référence ref = (D_ref, R_ref).
    # On parcourt les points par R croissant et on additionne des rectangles.
    pts = sorted({(p[0], p[1]) for p in front if p[0] <= ref[0] and p[1] <= ref[1]}, key=lambda p: (p[1], p[0]))
    hv, D_prec = 0.0, ref[0]
    for D, R in pts:
        if D < D_prec:
            hv += (D_prec - D) * (ref[1] - R)
            D_prec = D
    return hv


def grille_epsilon(N, nb=11):
    # Petites instances (n <= 12) : on teste toutes les valeurs 0, 1, ..., n.
    # Sinon 11 valeurs régulières de 0 à n (un point tous les 10 % de reports).
    n = len(N["interventions"])
    if n <= 12:
        return list(range(n + 1))
    return sorted(set(round(n * k / (nb - 1)) for k in range(nb)))


def epsilon_contrainte(S0, N, epsilons=None, temps_par_epsilon=15, graine=None, tau0=0.1):

    if epsilons is None:
        epsilons = grille_epsilon(N)
    M = 100 * N["penalite"]

    # T0 calculé une seule fois sur S0, puis gardé pour tous les epsilon
    random.seed(graine)
    T0 = calculer_T0(S0, N, tau0)

    resultats = []
    depart = S0

    for k, eps in enumerate(sorted(epsilons)):

        # coût pénalisé pour ce epsilon (eps=eps pour figer la valeur dans la fonction)
        def cout(D, R, eps=eps):
            return D + M * max(0, R - eps)

        g = None if graine is None else graine * 1000 + k
        S, valeur, infos = recuit_simule(depart, N, temps_max=temps_par_epsilon, graine=g, T0=T0,
                                         cout=cout, duree_cycle=0.97 * temps_par_epsilon)
        D, R = objectifs(S, N)
        resultats.append({"epsilon": eps, "D": D, "R": R, "respecte": R <= eps,
                          "temps": infos["temps"], "iterations": infos["iterations"], "solution": S})
        # chaînage : le epsilon suivant part de cette solution si elle respecte la contrainte
        if R <= eps:
            depart = S

    # on ne garde que les points qui respectent leur contrainte et qui ne sont pas dominés
    front = filtrer_non_domines([(r["D"], r["R"], r["solution"]) for r in resultats if r["respecte"]])
    return resultats, front

Exemple : le front obtenu sur N0.

In [ ]:
import epsilon_contrainte

S0_N0 = json.load(open("S0_communs.json"))["N0"]
resultats, front = epsilon_contrainte.epsilon_contrainte(S0_N0, N0, temps_par_epsilon=0.5, graine=1)
print("Front (D, R) :", [(p[0], p[1]) for p in front])

**Front exact de N0.** N0 n'a que 8 interventions : on peut énumérer toutes les solutions. Pour chaque technicien et chaque sous-ensemble d'interventions, on cherche le meilleur ordre de passage réalisable, puis on essaie toutes les affectations (4⁸ cas). On obtient le vrai front de Pareto, qui sert à contrôler nos méthodes.

In [ ]:
%%writefile code/exact_N0.py
# Résolution exacte de N0 par énumération complète (possible car N0 n'a que 8 interventions).
# On obtient le vrai front de Pareto (D, R) et l'optimum de f, pour vérifier nos méthodes.
# Principe :
#  1) pour chaque technicien et chaque sous-ensemble d'interventions, on teste tous les
#     ordres de passage et on garde la tournée réalisable la plus courte ;
#  2) on essaie toutes les affectations des 8 interventions (3 techniciens + reportée,
#     donc 4^8 = 65 536 cas) en réutilisant les tournées de l'étape 1.
# À lancer depuis le dossier du notebook : python code/exact_N0.py

import itertools, json
from pathlib import Path
Path("resultats").mkdir(exist_ok=True)
from socle import lire_instance, preparer_instance, verifier_tournee, distance_tournee

N = lire_instance("donnees/N0")
preparer_instance(N)
jobs, techs = N["interventions"]["intervention"].tolist(), N["techs"]

# étape 1 : meilleure tournée de chaque technicien pour chaque sous-ensemble
meilleure = {}
for t in techs:
    for r in range(len(jobs) + 1):
        for sous_ensemble in itertools.combinations(jobs, r):
            best = None
            for ordre in itertools.permutations(sous_ensemble):
                S = {t: list(ordre)}
                if verifier_tournee(S, N, t):
                    d = distance_tournee(N, t, list(ordre))
                    if best is None or d < best:
                        best = d
            # si aucun ordre n'est réalisable, ce sous-ensemble est impossible pour t
            if best is not None:
                meilleure[(t, frozenset(sous_ensemble))] = best

# étape 2 : toutes les affectations. affectation[i] = numéro du technicien de la mission i,
# et la valeur len(techs) veut dire "reportée"
min_D = {}     # pour chaque nombre de reports R, la plus petite distance trouvée
for affectation in itertools.product(range(len(techs) + 1), repeat=len(jobs)):
    D, ok = 0.0, True
    for k, t in enumerate(techs):
        v = meilleure.get((t, frozenset(j for j, a in zip(jobs, affectation) if a == k)))
        if v is None:
            ok = False
            break
        D += v
    if ok:
        R = sum(1 for a in affectation if a == len(techs))
        if R not in min_D or D < min_D[R]:
            min_D[R] = D

# on ne garde que les points non dominés : quand R augmente, D doit baisser strictement
front, D_prec = {}, float("inf")
for R in sorted(min_D):
    if min_D[R] < D_prec - 1e-9:
        front[R] = round(min_D[R], 3)
        D_prec = min_D[R]
print("front exact :", front)
print("optimum f = D + c.R :", round(min(D + N["penalite"] * R for R, D in min_D.items()), 3))
json.dump({"front": front}, open("resultats/N0_exact.json", "w"), indent=1)

### 5.2 Recherche locale de Pareto (PLS)

La PLS garde une archive des solutions non dominées. Elle prend une solution de l'archive, explore tout son voisinage (relocate, échange de deux interventions et inversion d'un morceau de tournée) et ajoute à l'archive chaque voisin non dominé, en retirant les points qu'il domine. Elle s'arrête quand toute l'archive a été explorée ou au bout du temps imparti. Les distances sont comptées en mètres entiers pour éviter les erreurs d'arrondi. Le fichier contient aussi l'hypervolume et la mesure de couverture entre deux fronts.

In [ ]:
%%writefile code/multiobjectif.py
# Partie Axel : recherche locale de Pareto (PLS) pour le problème biobjectif
# (minimiser la distance D et le nombre de reports R).
# Les distances sont comptées en mètres entiers (km x 1000) pour comparer les points
# sans problème d'arrondi. La PLS est déterministe : pas de hasard, une seule exécution.

import time, collections, math, itertools
from functools import lru_cache
from socle import *


def cle(S, N):
    # empreinte d'un planning, pour ne pas évaluer deux fois le même voisin
    return tuple(tuple(S[t]) for t in N['techs']) + (tuple(sorted(S['reportees'])),)


def copier(S):
    return {t: v[:] for t, v in S.items()}


def modele(N):
    # Prépare deux fonctions qui gardent en mémoire les tournées déjà calculées.
    dist = {k: int(round(v * 1000)) for k, v in N['dist'].items()}

    # route(t, seq) : distance en mètres de la tournée seq du technicien t,
    # ou None si elle est impossible. lru_cache évite de revérifier la même tournée.
    @lru_cache(maxsize=100000)
    def route(t, seq):
        if not seq:
            return 0
        if not verifier_tournee({t: list(seq)}, N, t):
            return None
        loc = N['domicile'][t]
        d = 0
        for m in seq:
            d += dist[loc, m]
            loc = m
        return d + dist[loc, N['domicile'][t]]

    # objectifs(S) : le couple (D en mètres, R), ou None si le planning est impossible
    def objectifs(S):
        ds = [route(t, tuple(S[t])) for t in N['techs']]
        if any(d is None for d in ds):
            return None
        return sum(ds), len(S['reportees'])

    return route, objectifs


def domine(p, q):
    # p domine q : pas pire sur les deux objectifs, et différent
    return p[0] <= q[0] and p[1] <= q[1] and p != q


def voisins(S, N, route, limite):
    """Relocate, insertion/report, swap et inversion d'un segment (2-opt)."""
    # C'est un générateur : il donne les voisins réalisables un par un (avec le nom du mouvement)
    # et s'arrête dès que l'heure limite est atteinte.
    listes = N['techs'] + ['reportees']
    vus = {cle(S, N)}

    def accepte(V, changes):
        # on ne revérifie que les tournées modifiées, et on refuse les voisins déjà vus
        if any(route(t, tuple(V[t])) is None for t in changes if t != 'reportees'):
            return False
        k = cle(V, N)
        if k in vus:
            return False
        vus.add(k)
        return True

    # 1) relocate : déplacer une mission (vers un autre technicien, une autre position ou les reports)
    for src in listes:
        for i, m in enumerate(S[src]):
            for dst in listes:
                if src == dst == 'reportees':
                    continue
                # le technicien de destination doit avoir la compétence
                if dst != 'reportees' and N['habilite'][dst][N['comp'][m]] != 1:
                    continue
                V0 = copier(S)
                V0[src].pop(i)
                positions = [len(V0[dst])] if dst == 'reportees' else range(len(V0[dst]) + 1)
                for pos in positions:
                    if time.perf_counter() >= limite:
                        return
                    V = copier(V0)
                    V[dst].insert(pos, m)
                    if accepte(V, {src, dst}):
                        yield V, 'relocate'

    # 2) swap : échanger deux missions (avec une reportée, ça fait un remplacement)
    places = [(t, i) for t in listes for i in range(len(S[t]))]
    for a, (t, i) in enumerate(places):
        for u, j in places[a + 1:]:
            if time.perf_counter() >= limite:
                return
            if t == u == 'reportees':
                continue
            V = copier(S)
            V[t][i], V[u][j] = V[u][j], V[t][i]
            if accepte(V, {t, u}):
                yield V, 'swap'

    # 3) 2-opt : inverser un morceau d'une tournée
    for t in N['techs']:
        for i in range(len(S[t])):
            for j in range(i + 2, len(S[t]) + 1):
                if time.perf_counter() >= limite:
                    return
                V = copier(S)
                V[t][i:j] = reversed(V[t][i:j])
                if accepte(V, {t}):
                    yield V, '2-opt'


def pls(S0, N, budget=180, solutions_initiales=None):
    # A : archive des solutions non dominées, rangées par leur point (D, R)
    # U : file des points de l'archive dont on n'a pas encore exploré le voisinage
    debut = time.perf_counter()
    limite = debut + budget
    route, objectifs = modele(N)
    A = {}
    U = collections.deque()
    trace = []
    evenements = []
    examines = 0
    explores = 0

    def ajouter(S, origine):
        # Essaie d'ajouter S à l'archive. Refusé s'il est impossible, déjà présent ou dominé.
        # S'il est ajouté, on retire de l'archive les points qu'il domine.
        nonlocal examines
        p = objectifs(S)
        examines += 1
        if p is None or p in A or any(domine(q, p) for q in A):
            return False
        elim = [q for q in A if domine(p, q)]
        for q in elim:
            del A[q]
        A[p] = copier(S)
        U.append(p)
        # on garde les 12 premiers événements pour illustrer le fonctionnement dans le rapport
        if len(evenements) < 12:
            evenements.append({'origine': origine, 'ajout': [p[0] / 1000, p[1]], 'retraits': [[q[0] / 1000, q[1]] for q in elim],
                               'archive': [[q[0] / 1000, q[1]] for q in sorted(A)]})
        trace.append({'temps': time.perf_counter() - debut, 'points': [[q[0] / 1000, q[1]] for q in sorted(A)]})
        return True

    for S in (solutions_initiales or [S0]):
        ajouter(S, 'initialisation')

    # Boucle principale : on prend un point de la file et on explore tout son voisinage
    while U and time.perf_counter() < limite:
        p = U.popleft()
        # il a pu être retiré de l'archive entre-temps (dominé par un nouveau point)
        if p not in A:
            continue
        S = A[p]
        complet = True
        for V, mvt in voisins(S, N, route, limite):
            if time.perf_counter() >= limite:
                complet = False
                break
            ajouter(V, mvt)
        if time.perf_counter() >= limite:
            complet = False
        if complet:
            explores += 1
        else:
            # budget fini au milieu de l'exploration : on remet le point dans la file et on arrête
            U.appendleft(p)
            break

    # le front final, trié par nombre de reports, avec D en km
    front = [{'D': p[0] / 1000, 'R': p[1], 'solution': S} for p, S in sorted(A.items(), key=lambda e: e[0][1])]
    return {'methode': 'PLS', 'budget': budget, 'temps': time.perf_counter() - debut, 'front': front,
            'arret': 'archive explorée' if not U else 'budget', 'voisins_evalues': examines, 'solutions_explorees': explores,
            'archive_trace': trace, 'illustration': evenements, 'initialisation': 'S0' if solutions_initiales is None else 'solutions fournies'}


def non_domines(points):
    # garde seulement les points non dominés (sans doublons)
    pts = sorted(set(tuple(p) for p in points))
    return [p for p in pts if not any(domine(q, p) for q in pts)]


def hypervolume(points, reference):
    # Aire dominée par les points et bornée par le point de référence (xmax, ymax).
    # Points triés par D croissant : on ajoute un rectangle à chaque fois que R descend.
    pts = non_domines(points)
    xmax, ymax = reference
    niveau = ymax
    aire = 0.0
    for x, y in sorted(pts):
        if x > xmax or y > ymax:
            continue
        if y < niveau:
            aire += (xmax - x) * (niveau - y)
            niveau = y
    return aire


def couverture(A, B):
    # C(A, B) : part des points de B qui sont dominés (ou égalés) par au moins un point de A
    return sum(any(a[0] <= b[0] + 1e-9 and a[1] <= b[1] for a in A) for b in B) / len(B) if B else None

Exemple : la PLS sur N0.

In [ ]:
import multiobjectif

res = multiobjectif.pls(S0_N0, N0, budget=5)
print("Front (D, R) :", [(p["D"], p["R"]) for p in res["front"]])

## 6. Campagnes d'expériences et analyses

Ces programmes réalisent les expériences présentées dans le rapport (cinq exécutions de 180 secondes par instance) puis refont les tableaux et les figures. Ils se lancent depuis un terminal, dans le dossier du notebook (par exemple `python code/campagne_recuit.py eps 0.1`).

**Recuit et ε-contrainte.** `campagne_recuit.py` lance l'analyse de sensibilité (un paramètre à la fois, puis des combinaisons), la campagne principale et la campagne ε-contrainte, et enregistre les résultats en JSON. `analyse_recuit.py` en tire les tableaux (meilleur coût, moyenne, écart-type, hypervolume) et les figures (convergence, sensibilité, fronts).

In [ ]:
%%writefile code/campagne_recuit.py
# Lance toutes les expériences du recuit et de l'epsilon-contrainte présentées dans le rapport.
# À lancer depuis le dossier du notebook :
#   python code/campagne_recuit.py sensi1                 -> analyse de sensibilité, étape 1 (30 s)
#   python code/campagne_recuit.py sensi2 '<combinaisons>'  -> étape 2 (60 s), combinaisons en json
#   python code/campagne_recuit.py mono '{"tau0": 0.1, "alpha": 0.99, "k_palier": 20}'
#   python code/campagne_recuit.py eps 0.1                 -> epsilon-contrainte (180 s par essai)
# Les résultats sont écrits dans resultats/ au format json (lus ensuite par analyse_recuit.py).

import sys, json, os, time
from multiprocessing import Pool
from epsilon_contrainte import *

INSTANCES = ["N0", "N10", "N20", "N30", "N50", "N80", "N100", "N150", "N200"]
GRAINES = [1, 2, 3, 4, 5]
BUDGETS = [60, 120, 180]
# réglage de départ, avant l'analyse de sensibilité
REGLAGE_INITIAL = {"tau0": 0.5, "alpha": 0.95, "k_palier": 5}
os.makedirs("resultats", exist_ok=True)

# on garde les instances déjà lues pour ne pas relire les csv à chaque essai
_cache = {}


def instance(nom):
    if nom not in _cache:
        N = lire_instance(os.path.join("donnees", nom))
        preparer_instance(N)
        _cache[nom] = (N, json.load(open("S0_communs.json"))[nom])
    return _cache[nom]


def tache_mono(a):
    # Un essai de 180 s du recuit sur une instance et une graine
    nom, g, reglage = a
    N, S0 = instance(nom)
    S, c, info = recuit_simule(S0, N, temps_max=max(BUDGETS), graine=g, garder_trace=True, **reglage)
    assert est_realisable(S, N)
    h = info["historique"]
    lignes = []
    # une seule exécution de 180 s donne aussi les résultats à 60 et 120 s :
    # on regarde la meilleure valeur connue à chaque instant dans l'historique
    for B in BUDGETS:
        pts = [x for x in h if x[0] <= B]
        lignes.append({"budget": B, "f": pts[-1][1], "temps_meilleur": pts[-1][0]})
    D, R = objectifs(S, N)
    tr = info["trace"]
    return {"instance": nom, "graine": g, "reglage": reglage, "lignes": lignes, "f": c, "D": D, "R": R,
            "iterations": info["iterations"], "acceptes": info["acceptes"], "T0": info["T0"],
            "palier": info["palier"], "rechauffages": info["rechauffages"], "temps_meilleur": info["temps_meilleur"],
            # on n'enregistre qu'environ 400 points de la trace pour limiter la taille du fichier
            "historique": h, "trace": tr[::max(1, len(tr) // 400)], "solution": S}


# Étape 1 de la sensibilité : on change un seul paramètre à la fois
SENSI_INST = ["N20", "N50", "N100", "N200"]
SENSI1 = [("base", {})] + [("tau0", {"tau0": v}) for v in (0.1, 0.9)] + \
         [("alpha", {"alpha": v}) for v in (0.80, 0.90, 0.99)] + [("k_palier", {"k_palier": v}) for v in (1, 20)]


def tache_sensi(a):
    # Un essai de sensibilité : réglage de base modifié par cfg
    nom, nomcfg, cfg, g, budget, base = a
    N, S0 = instance(nom)
    p = dict(base)
    p.update(cfg)
    S, c, info = recuit_simule(S0, N, temps_max=budget, graine=g, **p)
    return {"instance": nom, "parametre": nomcfg, **p, "graine": g, "budget": budget, "f": c, "T0": info["T0"],
            "iterations": info["iterations"], "rechauffages": info["rechauffages"],
            "temps_meilleur": info["temps_meilleur"], "historique": info["historique"]}


def tache_eps(a):
    # Un essai complet de l'epsilon-contrainte : 180 s partagées entre les valeurs de epsilon
    nom, g, tau0 = a
    N, S0 = instance(nom)
    grille = grille_epsilon(N)
    t = time.perf_counter()
    res, front = epsilon_contrainte(S0, N, epsilons=grille, temps_par_epsilon=180 / len(grille), graine=g, tau0=tau0)
    return {"instance": nom, "graine": g, "temps_total": time.perf_counter() - t, "bruts": res,
            "front": [{"D": p[0], "R": p[1], "solution": p[2]} for p in front]}


def lancer(fonction, taches, fichier):
    # On lance tous les essais et on sauvegarde les résultats à la fin
    out = []
    with Pool(4) as pool:
        for r in pool.imap_unordered(fonction, taches):
            out.append(r)
            print(fichier, len(out), "/", len(taches), r["instance"], flush=True)
    json.dump(out, open(os.path.join("resultats", fichier), "w"))


if __name__ == "__main__":
    quoi = sys.argv[1]
    # on commence par les grosses instances
    ordre = lambda nom: -INSTANCES.index(nom)
    if quoi == "sensi1":
        lancer(tache_sensi, [(i, n, c, g, 30, REGLAGE_INITIAL) for i in SENSI_INST for n, c in SENSI1 for g in GRAINES], "sensi_etape1.json")
    elif quoi == "sensi2":
        combos = json.loads(sys.argv[2])     # {"nom de la combinaison": {"tau0": ..., ...}, ...}
        lancer(tache_sensi, [(i, n, c, g, 60, REGLAGE_INITIAL) for i in ["N50", "N100", "N200"] for n, c in combos.items() for g in GRAINES], "sensi_etape2.json")
    elif quoi == "mono":
        reglage = json.loads(sys.argv[2])
        lancer(tache_mono, sorted([(i, g, reglage) for i in INSTANCES for g in GRAINES], key=lambda x: ordre(x[0])), "mono.json")
    elif quoi == "eps":
        tau0 = float(sys.argv[2])
        lancer(tache_eps, sorted([(i, g, tau0) for i in INSTANCES for g in GRAINES], key=lambda x: ordre(x[0])), "eps.json")

In [ ]:
%%writefile code/analyse_recuit.py
# Fabrique les tableaux et les figures du recuit et de l'epsilon-contrainte à partir
# des résultats déjà enregistrés dans resultats/*.json (rien n'est relancé, sauf la
# figure de dynamique sur N50, qui refait deux courts recuits).
# Sorties : resultats/tableaux/*.csv et resultats/figures/*.png
# À lancer depuis le dossier du notebook : python code/analyse_recuit.py

import json, os, statistics as st
import matplotlib
matplotlib.use("Agg")      # pas besoin d'écran, on enregistre seulement les images
import matplotlib.pyplot as plt
import pandas as pd
from epsilon_contrainte import *
from epsilon_contrainte import f, objectifs, hypervolume_2d, filtrer_non_domines

INST = ["N0", "N10", "N20", "N30", "N50", "N80", "N100", "N150", "N200"]

# couleurs des graphiques
BLEU, ORANGE, AQUA, JAUNE = "#2a78d6", "#eb6834", "#1baf7a", "#eda100"
GRIS, ENCRE, ENCRE2 = "#b8b7b2", "#0b0b0b", "#52514e"
os.makedirs("resultats/tableaux", exist_ok=True)
os.makedirs("resultats/figures", exist_ok=True)
plt.rcParams.update({"font.size": 9, "axes.edgecolor": GRIS, "axes.labelcolor": ENCRE2,
                     "xtick.color": ENCRE2, "ytick.color": ENCRE2, "axes.grid": True,
                     "grid.color": "#e6e5e0", "grid.linewidth": 0.6, "axes.spines.top": False,
                     "axes.spines.right": False, "lines.linewidth": 2, "figure.dpi": 150,
                     "axes.titlesize": 10, "axes.titleweight": "bold"})


def sd(x):
    # écart-type, 0 s'il n'y a qu'une valeur
    return st.stdev(x) if len(x) > 1 else 0.0


def ref_S0():
    # Pour chaque instance : les données, S0 et ses valeurs (sert de référence partout)
    out = {}
    for nom in INST:
        N = lire_instance(os.path.join("donnees", nom))
        preparer_instance(N)
        S0 = json.load(open("S0_communs.json"))[nom]
        D, R = objectifs(S0, N)
        out[nom] = {"N": N, "S0": S0, "f0": f(S0, N), "D0": D, "R0": R, "c": N["penalite"],
                    "n": len(N["interventions"]), "m": len(N["techs"])}
    return out


REF = ref_S0()


# ---------------------------------------------------------------- mono-objectif
def mono():
    if not os.path.exists("resultats/mono.json"):
        return None
    data = json.load(open("resultats/mono.json"))
    lignes, detail = [], []
    for nom in INST:
        runs = sorted([r for r in data if r["instance"] == nom], key=lambda r: r["graine"])
        f0 = REF[nom]["f0"]

        # tableau de synthèse : meilleure, moyenne, écart-type... pour chaque budget
        for B in (60, 120, 180):
            vals = [l["f"] for r in runs for l in r["lignes"] if l["budget"] == B]
            tb = [l["temps_meilleur"] for r in runs for l in r["lignes"] if l["budget"] == B]
            lignes.append({"Instance": nom, "n": REF[nom]["n"], "m": REF[nom]["m"], "Budget (s)": B,
                           "f(S0)": round(f0, 2), "Meilleure": round(min(vals), 2),
                           "Moyenne": round(st.mean(vals), 2), "Écart-type": round(sd(vals), 2),
                           "Pire": round(max(vals), 2),
                           "Amélioration meilleure (%)": round(100 * (f0 - min(vals)) / f0, 2),
                           "Amélioration moyenne (%)": round(100 * (f0 - st.mean(vals)) / f0, 2),
                           "Temps moyen au meilleur (s)": round(st.mean(tb), 1)})

        # détail de chaque exécution
        for r in runs:
            detail.append({"Instance": nom, "Graine": r["graine"], "f(60s)": round(r["lignes"][0]["f"], 2),
                           "f(120s)": round(r["lignes"][1]["f"], 2), "f(180s)": round(r["f"], 2),
                           "D": r["D"], "R": r["R"], "Itérations": r["iterations"],
                           "Acceptés (%)": round(100 * r["acceptes"] / r["iterations"], 1),
                           "T0": round(r["T0"], 2), "Palier": r["palier"], "Réchauffages": r["rechauffages"],
                           "Temps au meilleur (s)": round(r["temps_meilleur"], 1)})
    T = pd.DataFrame(lignes)
    Dt = pd.DataFrame(detail)
    T.to_csv("resultats/tableaux/recuit_synthese.csv", index=False, sep=";", decimal=",")
    Dt.to_csv("resultats/tableaux/recuit_detail_runs.csv", index=False, sep=";", decimal=",")

    # Figure de convergence : amélioration (%) du meilleur coût en fonction du temps,
    # moyenne des 5 graines avec la zone min-max
    fig, axs = plt.subplots(3, 3, figsize=(10, 8), sharex=True)
    # instants où on lit la courbe (plus serrés au début, où ça bouge beaucoup)
    grille = [0.01 * 1.06 ** k for k in range(200)]
    grille = [0.0] + [g for g in grille if g <= 180] + [180]
    for ax, nom in zip(axs.flat, INST):
        f0 = REF[nom]["f0"]
        courbes = []
        for r in [r for r in data if r["instance"] == nom]:
            h, c, k = r["historique"], [], 0
            # pour chaque instant de la grille, on prend la dernière amélioration connue
            for t in grille:
                while k + 1 < len(h) and h[k + 1][0] <= t:
                    k += 1
                c.append(100 * (f0 - h[k][1]) / f0)
            courbes.append(c)
        moy = [st.mean(v) for v in zip(*courbes)]
        ax.fill_between(grille, [min(v) for v in zip(*courbes)], [max(v) for v in zip(*courbes)],
                        color=BLEU, alpha=0.18, linewidth=0)
        ax.plot(grille, moy, color=BLEU)
        for B in (60, 120):
            ax.axvline(B, color=GRIS, linewidth=0.8, linestyle="--")
        ax.set_xscale("symlog", linthresh=1)
        ax.set_xlim(0, 180)
        ax.set_title(f"{nom}  (f(S0) = {f0:.1f})", loc="left")
        ax.set_xticks([0, 1, 10, 60, 180])
        ax.set_xticklabels(["0", "1", "10", "60", "180"])
    for ax in axs[-1]:
        ax.set_xlabel("Temps (s, échelle log)")
    for ax in axs[:, 0]:
        ax.set_ylabel("Amélioration vs S0 (%)")
    fig.suptitle("Recuit simulé — convergence du meilleur coût (moyenne et min–max sur 5 exécutions)",
                 x=0.01, ha="left", fontweight="bold")
    fig.tight_layout()
    fig.savefig("resultats/figures/recuit_convergence.png")
    plt.close(fig)

    # Dynamique d'une exécution sur N50 (20 s) : réglage initial contre réglage retenu.
    # C'est le seul endroit où on relance un calcul (2 recuits de 20 s).
    N, S0 = REF["N50"]["N"], REF["N50"]["S0"]
    fig, axs = plt.subplots(2, 2, figsize=(11, 5.5), sharex=True, sharey="row",
                            gridspec_kw={"height_ratios": [2, 1]})
    for col, (titre, p) in enumerate([("Réglage initial : τ0=0,5  α=0,95  k=5", dict(tau0=0.5, alpha=0.95, k_palier=5)),
                                      ("Réglage retenu : τ0=0,1  α=0,99  k=20", dict(tau0=0.1, alpha=0.99, k_palier=20))]):
        _, _, info = recuit_simule(S0, N, temps_max=20, graine=1, garder_trace=True, **p)
        tr, hb = info["trace"], info["historique"]
        a1, a2 = axs[0, col], axs[1, col]
        # en haut : coût courant et meilleur coût ; en bas : la température
        a1.plot([q[0] for q in tr], [q[1] for q in tr], color=GRIS, linewidth=0.7, label="coût courant")
        a1.step([h[0] for h in hb], [h[1] for h in hb], where="post", color=BLEU, label="meilleur coût")
        a1.set_title(f"{titre}\n{info['rechauffages']} réchauffage(s), meilleur = {hb[-1][1]:.1f}", loc="left")
        a2.plot([q[0] for q in tr], [q[2] for q in tr], color=ORANGE, linewidth=1)
        a2.set_yscale("log")
        a2.set_xlabel("Temps (s)")
    axs[0, 0].set_ylabel("f = D + c·R")
    axs[1, 0].set_ylabel("Température T")
    axs[0, 0].legend(frameon=False)
    fig.suptitle("N50 — dynamique d'une exécution de 20 s", x=0.01, ha="left", fontweight="bold")
    fig.tight_layout()
    fig.savefig("resultats/figures/recuit_dynamique_N50.png")
    plt.close(fig)

    # Réglage initial contre réglage retenu (mêmes graines, mêmes budgets), si le fichier existe
    if os.path.exists("resultats/mono_reglage_initial.json"):
        ini = json.load(open("resultats/mono_reglage_initial.json"))
        comp = []
        for nom in INST:
            for B in (60, 120, 180):
                a = [l["f"] for r in ini if r["instance"] == nom for l in r["lignes"] if l["budget"] == B]
                b = [l["f"] for r in data if r["instance"] == nom for l in r["lignes"] if l["budget"] == B]
                comp.append({"Instance": nom, "Budget (s)": B, "Initial moyenne": round(st.mean(a), 2),
                             "Initial meilleure": round(min(a), 2), "Retenu moyenne": round(st.mean(b), 2),
                             "Retenu meilleure": round(min(b), 2),
                             "Gain moyen (%)": round(100 * (st.mean(a) - st.mean(b)) / st.mean(a), 2)})
        pd.DataFrame(comp).to_csv("resultats/tableaux/recuit_reglage_initial_vs_retenu.csv", index=False, sep=";", decimal=",")
    return T, Dt


# ---------------------------------------------------------------- sensibilité
def sensi(best_connu):
    # best_connu : meilleur coût connu de chaque instance, pour calculer un écart en %
    if not os.path.exists("resultats/sensi_etape1.json"):
        return None
    data = json.load(open("resultats/sensi_etape1.json"))
    df = pd.DataFrame([{k: v for k, v in r.items() if k != "historique"} for r in data])
    df["ecart"] = df.apply(lambda r: 100 * (r["f"] - best_connu[r["instance"]]) / best_connu[r["instance"]], axis=1)
    agg = df.groupby(["parametre", "tau0", "alpha", "k_palier", "instance"]).agg(
        Meilleure=("f", "min"), Moyenne=("f", "mean"), Ecart_type=("f", "std"),
        Ecart_moyen_pct=("ecart", "mean"), T0=("T0", "mean"), Iterations=("iterations", "mean"),
        Rechauffes=("rechauffages", "mean"), Temps_meilleur=("temps_meilleur", "mean")).round(2).reset_index()
    agg.to_csv("resultats/tableaux/recuit_sensibilite.csv", index=False, sep=";", decimal=",")

    # une courbe par paramètre (les deux autres restent au réglage initial)
    base = df[df.parametre == "base"]
    fig, axs = plt.subplots(1, 3, figsize=(11, 3.6), sharey=True)
    couleurs = dict(zip(["N20", "N50", "N100", "N200"], [BLEU, ORANGE, AQUA, JAUNE]))
    for ax, (p, lib) in zip(axs, [("tau0", "τ0 (taux d'acceptation initial → T0)"),
                                  ("alpha", "α (coefficient de refroidissement)"),
                                  ("k_palier", "k (longueur du palier = k × n)")]):
        sous = pd.concat([base, df[df.parametre == p]])
        for nom, coul in couleurs.items():
            g = sous[sous.instance == nom].groupby(p)["ecart"].agg(["mean", "std"]).reset_index()
            ax.errorbar(g[p], g["mean"], yerr=g["std"], color=coul, marker="o", markersize=5, capsize=3,
                        linewidth=1.6, label=nom)
        ax.set_xlabel(lib)
        if p == "k_palier":
            ax.set_xscale("log")
            ax.set_xticks([1, 5, 20])
            ax.set_xticklabels(["1", "5", "20"])
    axs[0].set_ylabel("Écart moyen au meilleur connu (%)")
    axs[0].legend(frameon=False, title="Instance")
    fig.suptitle("Sensibilité du recuit, étape 1 (30 s, 5 graines ; un paramètre varie, les autres au réglage initial τ0=0,5 α=0,95 k=5)",
                 x=0.01, ha="left", fontweight="bold")
    fig.tight_layout()
    fig.savefig("resultats/figures/recuit_sensibilite.png")
    plt.close(fig)

    # étape 2 : les combinaisons de paramètres (60 s)
    if os.path.exists("resultats/sensi_etape2.json"):
        d2 = pd.DataFrame([{k: v for k, v in r.items() if k != "historique"} for r in json.load(open("resultats/sensi_etape2.json"))])
        d2.groupby(["instance", "parametre"]).agg(Meilleure=("f", "min"), Moyenne=("f", "mean"), Ecart_type=("f", "std"),
                                               Rechauffes=("rechauffages", "mean"), Temps_meilleur=("temps_meilleur", "mean")
                                               ).round(2).reset_index().to_csv(
            "resultats/tableaux/recuit_sensibilite_etape2.csv", index=False, sep=";", decimal=",")
    return agg


# ---------------------------------------------------------------- epsilon-contrainte
def eps(mono_best_pts):
    # mono_best_pts : meilleur point (D, R) du recuit mono-objectif, affiché sur les fronts
    if not os.path.exists("resultats/eps.json"):
        return None
    data = json.load(open("resultats/eps.json"))
    lignes, par_eps = [], []
    # front exact de N0 (calculé par exact_N0.py), pour comparer
    exact = json.load(open("resultats/N0_exact.json")) if os.path.exists("resultats/N0_exact.json") else None
    fig, axs = plt.subplots(3, 3, figsize=(11, 9))
    fronts_union = {}
    for ax, nom in zip(axs.flat, INST):
        runs = sorted([r for r in data if r["instance"] == nom], key=lambda r: r["graine"])
        n = REF[nom]["n"]
        # point de référence de l'hypervolume : un peu moins bon que tous les points
        # (10 % de marge sur la distance de S0, et n + 1 reports)
        ref = (REF[nom]["D0"] * 1.1 + 1, n + 1)
        hvs = [hypervolume_2d([(p["D"], p["R"]) for p in r["front"]], ref) for r in runs]
        # union des fronts des 5 graines
        union = filtrer_non_domines([(p["D"], p["R"], None) for r in runs for p in r["front"]])
        fronts_union[nom] = [(p[0], p[1]) for p in union]
        lignes.append({"Instance": nom, "Nb ε": len(runs[0]["bruts"]),
                       "Temps par exécution (s)": round(st.mean(r["temps_total"] for r in runs), 1),
                       "Taille front (moy.)": round(st.mean(len(r["front"]) for r in runs), 1),
                       "Taille front union": len(union),
                       "Hypervolume moyen": round(st.mean(hvs), 1), "Hypervolume écart-type": round(sd(hvs), 1),
                       "Hypervolume union": round(hypervolume_2d(fronts_union[nom], ref), 1),
                       "Point de référence (D, R)": f"({ref[0]:.1f}, {ref[1]})"})

        # détail pour chaque valeur de epsilon
        for k in range(len(runs[0]["bruts"])):
            bs = [r["bruts"][k] for r in runs]
            ok = [b for b in bs if b["respecte"]]
            par_eps.append({"Instance": nom, "ε": bs[0]["epsilon"], "Exécutions respectant ε": f"{len(ok)}/5",
                            "D meilleure": round(min(b["D"] for b in ok), 2) if ok else None,
                            "D moyenne": round(st.mean(b["D"] for b in ok), 2) if ok else None,
                            "D écart-type": round(sd([b["D"] for b in ok]), 2) if ok else None,
                            "R obtenu (min–max)": f"{min(b['R'] for b in bs)}–{max(b['R'] for b in bs)}",
                            "Temps (s)": round(st.mean(b["temps"] for b in bs), 1)})

        # graphique du front
        if nom == "N0" and exact:
            ex = sorted((float(D), int(R)) for R, D in exact["front"].items())
            ax.scatter([p[1] for p in ex], [p[0] for p in ex], s=110, facecolors="none", edgecolors=ENCRE2,
                       linewidths=1.2, label="front exact", zorder=2)
        pts = sorted(fronts_union[nom], key=lambda p: p[1])
        ax.plot([p[1] for p in pts], [p[0] for p in pts], color=BLEU, marker="o", markersize=5,
                linewidth=1.2, label="ε-contrainte (union 5 graines)", zorder=3)
        if nom in mono_best_pts:
            D, R = mono_best_pts[nom]
            ax.scatter([R], [D], marker="D", s=50, color=ORANGE, edgecolors="white", linewidths=1.5,
                       label="recuit mono-objectif (meilleur)", zorder=4)
        ax.scatter([REF[nom]["R0"]], [REF[nom]["D0"]], marker="s", s=40, color=GRIS, edgecolors="white",
                   label="heuristique S0", zorder=4)
        ax.set_title(nom, loc="left")
    for ax in axs[-1]:
        ax.set_xlabel("R : nombre de reports")
    for ax in axs[:, 0]:
        ax.set_ylabel("D : distance (km)")
    h, l = axs.flat[0].get_legend_handles_labels()
    fig.legend(h, l, loc="lower center", ncol=4, frameon=False)
    fig.suptitle("Méthode ε-contrainte (min D s.c. R ≤ ε) — fronts non dominés obtenus",
                 x=0.01, ha="left", fontweight="bold")
    fig.tight_layout(rect=(0, 0.04, 1, 1))
    fig.savefig("resultats/figures/epsilon_fronts.png")
    plt.close(fig)
    T = pd.DataFrame(lignes)
    P = pd.DataFrame(par_eps)
    T.to_csv("resultats/tableaux/epsilon_synthese.csv", index=False, sep=";", decimal=",")
    P.to_csv("resultats/tableaux/epsilon_par_valeur.csv", index=False, sep=";", decimal=",")
    # ce fichier sert aussi à la comparaison avec la PLS
    json.dump(fronts_union, open("resultats/tableaux/epsilon_fronts_union.json", "w"), indent=1)
    return T, P


if __name__ == "__main__":
    pd.set_option("display.width", 250)
    pd.set_option("display.max_columns", 30)
    m = mono()
    best_connu, best_pts = {}, {}
    if m:
        data = json.load(open("resultats/mono.json"))
        # meilleure solution connue de chaque instance (sert à la sensibilité et aux fronts)
        for nom in INST:
            r = min((r for r in data if r["instance"] == nom), key=lambda r: r["f"])
            best_connu[nom], best_pts[nom] = r["f"], (r["D"], r["R"])
        print(m[0].to_string(index=False))
    if os.path.exists("resultats/sensi_etape1.json"):
        sd_ = json.load(open("resultats/sensi_etape1.json"))
        if os.path.exists("resultats/sensi_etape2.json"):
            sd_ += json.load(open("resultats/sensi_etape2.json"))
        # on complète le meilleur connu avec les essais de sensibilité
        for nom in set(r["instance"] for r in sd_):
            best_connu[nom] = min([best_connu.get(nom, 1e18)] + [r["f"] for r in sd_ if r["instance"] == nom])
        print(sensi(best_connu).to_string(index=False))
    e = eps(best_pts)
    if e:
        print(e[0].to_string(index=False))

**Génétique et PLS.** `lancer.py` lance le génétique ou la PLS sur les instances choisies (`python code/lancer.py genetique --instances N50 --secondes 180`). `sensibilite_diversite.py` réalise le réglage du génétique (un paramètre à la fois, combinaisons, et test avec ou sans les mécanismes de diversité). `analyser.py` refait les tableaux et figures du génétique et de la comparaison PLS / ε-contrainte.

In [ ]:
%%writefile code/lancer.py
"""Lanceur de compilation. Les algorithmes finaux importés restent inchangés."""
# Lance le génétique ou la PLS sur une ou plusieurs instances et enregistre le résultat en json.
# Exemples (depuis le dossier du notebook) :
#   python code/lancer.py genetique --instances N0 --secondes 1 --graines 1
#   python code/lancer.py pls --instances N50 --secondes 180
# Les nouveaux résultats vont dans nouveaux_resultats/ (les anciens ne sont jamais écrasés).

import argparse, json, time
from pathlib import Path
from socle import lire_instance, preparer_instance, est_realisable, f, heuristique_gloutonne
from genetique import algorithme_genetique
from multiobjectif import pls

# ROOT = dossier du notebook (le dossier au-dessus de code/)
ROOT = Path(__file__).resolve().parent.parent
NOMS = ['N0', 'N10', 'N20', 'N30', 'N50', 'N80', 'N100', 'N150', 'N200']


def charger(nom, depart='commun'):
    # Lit l'instance et choisit la solution de départ :
    #  - 'commun' : le S0 commun du groupe (S0_communs.json)
    #  - 'historique-pls' : le départ utilisé pour les archives PLS du rapport
    #  - 'exploratoire' : départ historique N50 sauvegardé ; glouton recalculé ailleurs
    N = lire_instance(str(ROOT / 'donnees' / nom))
    preparer_instance(N)
    if depart == 'historique-pls':
        S0 = json.loads((ROOT / 'resultats/multiobjectif' / f'PLS_{nom}.json').read_text())['solution_initiale']
    elif depart == 'exploratoire':
        historiques = json.loads((ROOT / 'S0_exploratoires.json').read_text())
        S0 = historiques[nom] if nom in historiques else heuristique_gloutonne(N)
    else:
        S0 = json.loads((ROOT / 'S0_communs.json').read_text())[nom]
    if not est_realisable(S0, N):
        raise ValueError('S0 non réalisable')
    return N, S0


def main():
    # lecture des options de la ligne de commande
    p = argparse.ArgumentParser(description=__doc__)
    p.add_argument('methode', choices=['verifier', 'genetique', 'pls'])
    p.add_argument('--instances', nargs='+', choices=NOMS, default=['N0'])
    p.add_argument('--secondes', type=float, default=180)
    p.add_argument('--graines', nargs='+', type=int, default=[1, 2, 3, 4, 5])
    p.add_argument('--sortie', default='nouveaux_resultats')
    p.add_argument('--depart', choices=['commun', 'historique-pls'], default='commun')
    a = p.parse_args()
    if a.secondes <= 0:
        p.error('--secondes doit être positif')
    out = ROOT / a.sortie

    for nom in a.instances:
        N, S0 = charger(nom, a.depart)
        print(nom, 'S0 =', round(f(S0, N), 3), flush=True)
        # "verifier" : on contrôle seulement que S0 se charge et est réalisable
        if a.methode == 'verifier':
            continue
        if a.methode == 'genetique':
            # on relève aussi les coûts à 60 et 120 s s'ils sont avant la fin du budget
            budgets = tuple(b for b in (60, 120) if b < a.secondes) + (a.secondes,)
            rr, ss = algorithme_genetique(S0, N, budgets=budgets, graines=tuple(a.graines))
            res = {'instance': nom, 'methode': 'AG', 'solution_initiale': S0, 'reference': f(S0, N), 'resultats': rr, 'solutions': ss}
        else:
            res = {'instance': nom, 'solution_initiale': S0, 'reference': f(S0, N), **pls(S0, N, budget=a.secondes)}
        out.mkdir(parents=True, exist_ok=True)
        # nom de fichier horodaté pour ne rien écraser
        path = out / f'{a.methode}_{nom}_{time.time_ns()}.json'
        path.write_text(json.dumps(res, ensure_ascii=False, indent=2))
        print('Sauvegardé :', path, flush=True)


if __name__ == '__main__':
    main()

In [ ]:
%%writefile code/sensibilite_diversite.py
"""Lanceur de compilation des plans exploratoires utilisés, avec le génétique final.
Les sources des plans d'origine figurent dans sources_protocoles_utilises/.
Les S0 exploratoires sont recalculés par le glouton historique, comme lors des
réglages. Ces essais ne doivent pas être mélangés à la campagne principale.
"""
# Trois plans d'expériences pour le génétique (depuis le dossier du notebook) :
#   python code/sensibilite_diversite.py sensibilite    -> un paramètre à la fois (30 s)
#   python code/sensibilite_diversite.py combinaisons   -> quelques combinaisons (30 s)
#   python code/sensibilite_diversite.py diversite      -> test d'ablation avec/sans diversité (20 s)
# Pour un essai rapide : ajouter --instances N50 --graines 1 --secondes 1

import argparse, json, time
from pathlib import Path
from lancer import charger, ROOT
from genetique import algorithme_genetique
from socle import f

# réglage final de référence
BASE = dict(taille_population=20, proba_croisement=.8, proba_mutation=.25)


def plans(phase):
    # Renvoie la liste des réglages à tester : (nom, paramètres, diversité activée ou non)
    if phase == 'diversite':
        return [('sans', BASE.copy(), False), ('avec', BASE.copy(), True)]
    if phase == 'combinaisons':
        return [('reference', BASE.copy(), True)] + [(f'P{n}_pc1_pm01', dict(taille_population=n, proba_croisement=1., proba_mutation=.1), True) for n in [15, 20, 30]]
    # phase 'sensibilite' : on change une seule valeur à la fois par rapport à BASE
    result = [('reference', BASE.copy(), True)]
    valeurs = {'taille_population': [5, 10, 40, 80], 'proba_croisement': [0., .5, 1.], 'proba_mutation': [0., .1, .5, 1.]}
    for param, vals in valeurs.items():
        for v in vals:
            config = BASE.copy()
            config[param] = v
            result.append((f'{param}_{v}', config, True))
    return result


def main():
    p = argparse.ArgumentParser(description=__doc__)
    p.add_argument('phase', choices=['sensibilite', 'combinaisons', 'diversite'])
    p.add_argument('--instances', nargs='+', choices=['N20', 'N50', 'N100', 'N200'])
    p.add_argument('--graines', nargs='+', type=int, default=[1, 2, 3, 4, 5])
    p.add_argument('--secondes', type=float)
    p.add_argument('--sortie', default='nouveaux_essais_exploratoires')
    a = p.parse_args()
    # budgets d'origine : 20 s pour la diversité, 30 s pour le reste
    budget = a.secondes if a.secondes is not None else (20 if a.phase == 'diversite' else 30)
    if budget <= 0:
        p.error('Le budget doit être positif')
    noms = a.instances or (['N50'] if a.phase == 'diversite' else ['N50', 'N200'])
    out = ROOT / a.sortie
    out.mkdir(parents=True, exist_ok=True)

    for nom in noms:
        N, S0 = charger(nom, 'exploratoire')
        for label, config, active in plans(a.phase):
            for seed in a.graines:
                print(a.phase, nom, label, 'graine', seed, flush=True)
                rr, ss = algorithme_genetique(S0, N, **config, budgets=(budget,), graines=(seed,), diversite=active)
                obj = {'phase': a.phase, 'nom': nom, 'reglage': label, 'graine': seed, 'budget': budget, 'parametres': config, 'diversite': active, 'reference': f(S0, N), 'solution_initiale': S0, 'resultat': rr[0], 'solution': ss[0]}
                # un fichier par essai
                dest = out / f'{a.phase}_{nom}_{label}_{seed}_{time.time_ns()}.json'
                dest.write_text(json.dumps(obj, ensure_ascii=False, indent=2))


if __name__ == '__main__':
    main()

In [ ]:
%%writefile code/analyser.py
"""Analyse des résultats sauvegardés, sans relancer les algorithmes.
Lanceur regroupé pour la compilation : statistiques AG, convergence, comparaison
PLS/epsilon et sensibilité. Les résultats livrés sont toujours lus sans mutation.
"""
# Refait les tableaux et figures du génétique et de la comparaison PLS / epsilon
# à partir des fichiers de resultats/. Depuis le dossier du notebook :
#   python code/analyser.py
# Sorties dans analyses_regenerees/ (convergence_AG.png, synthese_AG.csv, ...).

import argparse, json, csv, statistics
from pathlib import Path
ROOT = Path(__file__).resolve().parent.parent
NOMS = ['N0', 'N10', 'N20', 'N30', 'N50', 'N80', 'N100', 'N150', 'N200']


def ecrire(path, rows):
    # écrit une liste de dictionnaires dans un csv (séparateur ; pour Excel en français)
    if not rows:
        return
    with path.open('w', encoding='utf-8-sig', newline='') as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0]), delimiter=';')
        w.writeheader()
        w.writerows(rows)


def main():
    p = argparse.ArgumentParser(description=__doc__)
    p.add_argument('--sortie', default='analyses_regenerees')
    a = p.parse_args()
    out = ROOT / a.sortie
    out.mkdir(parents=True, exist_ok=True)
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    from multiobjectif import hypervolume, couverture
    plt.rcParams.update({'font.size': 10, 'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True, 'grid.alpha': .2})

    # 1) Génétique : statistiques à 180 s et courbes de convergence (une case par instance)
    resume = []
    fig, axes = plt.subplots(3, 3, figsize=(12, 9), layout='constrained')
    for ax, n in zip(axes.flat, NOMS):
        runs = [json.loads(p.read_text()) for p in sorted((ROOT / 'resultats/genetique').glob(f'{n}_graine_*.json'))]
        vals = [r['resultat']['Coût 180'] for r in runs]
        ref = runs[0]['reference']
        resume.append(dict(Instance=n, Meilleur=min(vals), Moyenne=statistics.mean(vals), SD=statistics.stdev(vals), Gain_moyen_pct=100 * (ref - statistics.mean(vals)) / ref, Generations=statistics.mean(r['resultat']['Générations'] for r in runs)))
        for r in runs:
            # on prolonge l'historique jusqu'à 180 s pour que la courbe aille au bout
            h = r['resultat']['Historique'] + [[180, r['resultat']['Coût 180']]]
            ax.step([t for t, c in h], [c for t, c in h], where='post', alpha=.65, lw=1)
        ax.set_title(n)
        ax.set_xlabel('Temps (s)')
        ax.set_ylabel('Meilleur coût pénalisé')
    fig.savefig(out / 'convergence_AG.png', dpi=200)
    plt.close(fig)
    ecrire(out / 'synthese_AG.csv', resume)

    # 2) Comparaison des fronts PLS et epsilon-contrainte (hypervolume et couverture)
    eps = json.loads((ROOT / 'resultats/epsilon_fourni.json').read_text())
    comp = []
    fig, axes = plt.subplots(3, 3, figsize=(12, 9), layout='constrained')
    for ax, n in zip(axes.flat, NOMS):
        archive = json.loads((ROOT / 'resultats/multiobjectif' / f'PLS_{n}.json').read_text())
        # Même précision que les points epsilon publiés et le comparateur original.
        P = [(round(x['D'], 1), x['R']) for x in archive['front']]
        E = [tuple(x) for x in eps['fronts_union'][n]]
        ref = eps['references_HV_affichees'][n]
        for pts, color, marker, label in [(P, '#266480', 'o', 'PLS : 1 exécution'), (E, '#c57230', 's', 'ε : union de 5 exécutions')]:
            pts = sorted(pts)
            ax.plot([x for x, y in pts], [y for x, y in pts], color=color, marker=marker, ms=3, lw=.8, label=label)
        ax.set_title(n)
        ax.set_xlabel('Distance D (km)')
        ax.set_ylabel('Reports R')
        comp.append(dict(Instance=n, Points_PLS=len(P), Points_epsilon=len(E), HV_PLS=hypervolume(P, ref), HV_epsilon=hypervolume(E, ref), Couverture_PLS_epsilon_pct=100 * couverture(P, E), Couverture_epsilon_PLS_pct=100 * couverture(E, P), Temps_PLS_s=archive['temps'], Budget_epsilon_cumule_s=900))
    h, l = axes.flat[0].get_legend_handles_labels()
    fig.legend(h, l, loc='outside lower center', ncol=2)
    fig.suptitle('Archives obtenues : conditions et budgets cumulés différents')
    fig.savefig(out / 'comparaison_PLS_epsilon.png', dpi=200)
    plt.close(fig)
    ecrire(out / 'comparaison_PLS_epsilon.csv', comp)

    # 3) Sensibilité du génétique (coût moyen +- écart-type pour chaque valeur testée)
    with (ROOT / 'resultats/sensibilite_etendue.csv').open(encoding='utf-8-sig') as f:
        rows = list(csv.DictReader(f, delimiter=';'))
    fig, axes = plt.subplots(2, 3, figsize=(12, 7), layout='constrained')
    for line, n in zip(axes, ['N50', 'N200']):
        for ax, param in zip(line, ['taille_population', 'proba_croisement', 'proba_mutation']):
            r = sorted([x for x in rows if x['Instance'] == n and x['Parametre'] == param], key=lambda x: float(x['Valeur']))
            ax.errorbar([float(x['Valeur']) for x in r], [float(x['Moyenne']) for x in r], yerr=[float(x['SD']) for x in r], marker='o', capsize=3)
            ax.set_title(f'{n} : {param}')
            ax.set_xlabel('Valeur')
            ax.set_ylabel('Coût moyen ± écart-type')
    fig.savefig(out / 'sensibilite_AG.png', dpi=200)
    plt.close(fig)
    print('Analyses produites dans', out)


if __name__ == '__main__':
    main()

**Contrôle indépendant.** `verification_csv.py` recalcule le coût d'une solution en relisant directement les fichiers CSV, avec un code écrit séparément du socle ; une contrainte non respectée arrête le programme. `verifier.py` l'applique aux résultats du génétique et de la PLS, puis lance des essais très courts des deux méthodes.

In [ ]:
%%writefile code/verification_csv.py
"""Contrôles indépendants des résultats, directement sur les CSV."""
# Recalcule le coût d'une solution en relisant les csv avec le module csv (sans pandas
# et sans les fonctions du socle). Si une contrainte n'est pas respectée, assert s'arrête.
# Ça permet de vérifier nos résultats avec un code écrit séparément.

import csv, json, hashlib, math
from pathlib import Path
from collections import Counter
ROOT = Path(__file__).resolve().parent.parent


def lignes(path):
    with path.open(encoding='utf-8-sig') as entree:
        return list(csv.DictReader(entree, delimiter=';'))


def verifier(S, nom):
    # Lecture des données de l'instance
    base = ROOT / 'donnees' / nom
    techs = {r['technicien']: r for r in lignes(base / 'techniciens.csv')}
    jobs = {r['intervention']: r for r in lignes(base / 'interventions.csv')}
    trajets = {(r['origine'], r['destination']): (float(r['temps_min']), float(r['distance_km'])) for r in lignes(base / 'trajets.csv')}
    dispos = {t: [] for t in techs}
    for r in lignes(base / 'disponibilites.csv'):
        dispos[r['technicien']].append((float(r['debut_min']), float(r['fin_min'])))
    for a in dispos.values():
        a.sort()
    c = float(next(r['valeur'] for r in lignes(base / 'parametres.csv') if r['parametre'] == 'cout_report'))

    # chaque intervention apparaît exactement une fois
    assert set(S) == set(techs) | {'reportees'}
    assert Counter([m for t in techs for m in S[t]] + S['reportees']) == Counter(jobs.keys())
    total = 0

    def arriver(depart, duree, plages):
        # heure d'arrivée d'un trajet qui ne roule que pendant les créneaux
        reste = duree
        if reste <= 1e-9:
            return depart
        for a, b in plages:
            debut = max(a, depart)
            if debut >= b:
                continue
            disponible = b - debut
            if reste <= disponible + 1e-9:
                return debut + reste
            reste -= disponible
        raise AssertionError('Trajet hors disponibilités')

    # on refait chaque tournée minute par minute
    for t, info in techs.items():
        if not S[t]:
            continue
        plages = dispos[t]
        heure = plages[0][0]
        lieu = info['domicile']
        for m in S[t]:
            j = jobs[m]
            assert int(info[j['competence_requise']]) == 1
            duree, distance = trajets[(lieu, m)]
            heure = arriver(heure, duree, plages)
            total += distance
            duree = float(j['duree_min'])
            w0 = float(j['debut_fenetre_min'])
            w1 = float(j['fin_fenetre_min'])
            # débuts possibles : la mission doit tenir dans un créneau et dans sa fenêtre
            debuts = [max(heure, w0, a) for a, b in plages if max(heure, w0, a) + duree <= min(b, w1) + 1e-9]
            assert debuts, 'Intervention hors créneau ou coupure'
            heure = min(debuts) + duree
            lieu = m
        # retour au domicile
        duree, distance = trajets[(lieu, info['domicile'])]
        arriver(heure, duree, plages)
        total += distance
    return total + c * len(S['reportees'])

In [ ]:
%%writefile code/verifier.py
"""Contrôle des sorties livrées, puis essai court des algorithmes finaux."""
# Depuis le dossier du notebook : python code/verifier.py
# 1) relit les 45 résultats du génétique et les 9 archives PLS et recalcule leurs coûts
#    avec verification_csv.py ;
# 2) lance des essais très courts du génétique et de la PLS pour vérifier que tout marche.

import json, math, functools
from pathlib import Path
from collections import Counter
from lancer import ROOT, NOMS, charger
from genetique import algorithme_genetique
from multiobjectif import pls, domine
import verification_csv as controle
# Les CSV restent en mémoire ; la logique du contrôleur indépendant est inchangée.
controle.lignes = functools.lru_cache(maxsize=None)(controle.lignes)


def main():
    ag = points = 0
    for n in NOMS:
        N, S0 = charger(n)
        # résultats du génétique : coût recalculé = coût enregistré, et pas pire que S0
        for p in (ROOT / 'resultats/genetique').glob(f'{n}_graine_*.json'):
            r = json.loads(p.read_text())
            val = controle.verifier(r['solution'], n)
            assert math.isclose(val, r['resultat']['Coût 180'], abs_tol=1e-7), p
            assert val <= r['reference'] + 1e-7, p
            ag += 1
        # archive PLS : aucun point dominé, pas de doublon, coûts justes
        r = json.loads((ROOT / 'resultats/multiobjectif' / f'PLS_{n}.json').read_text())
        front = r['front']
        assert all(not domine((b['D'], b['R']), (a['D'], a['R'])) for a in front for b in front)
        assert len({(a['D'], a['R']) for a in front}) == len(front)
        for a in front:
            val = controle.verifier(a['solution'], n)
            assert math.isclose(val - N['penalite'] * a['R'], a['D'], abs_tol=1e-7), (n, a['D'], a['R'])
            assert len(a['solution']['reportees']) == a['R']
            points += 1
    assert ag == 45
    print(f'{ag} sorties AG et {points} points PLS : coûts, faisabilité et non-dominance vérifiés.', flush=True)

    # essais courts : le génétique ne doit pas modifier S0, et le coût à 0,1 s
    # ne doit pas être pire que celui à 0,05 s
    for n in ['N0', 'N50', 'N200']:
        N, S = charger(n)
        avant = json.dumps(S, sort_keys=True)
        rr, ss = algorithme_genetique(S, N, budgets=(.05, .1), graines=(1,))
        assert json.dumps(S, sort_keys=True) == avant
        assert math.isclose(controle.verifier(ss[0], n), rr[0]['Coût 0.1'], abs_tol=1e-7)
        assert rr[0]['Coût 0.05'] >= rr[0]['Coût 0.1'] - 1e-7
    # la PLS doit retrouver les 8 points du front de N0 en 1 s
    N, S = charger('N0', 'historique-pls')
    r = pls(S, N, budget=1)
    assert len(r['front']) == 8
    print('Essais courts AG (N0, N50, N200) et PLS (N0) : OK.')


if __name__ == '__main__':
    main()

## 7. Compléments ajoutés après la soutenance

### 7.1 Recuit simulé : P essais indépendants

Version du recuit qui suit le schéma du cours : P essais indépendants, chacun depuis une nouvelle solution aléatoire (obtenue par n mouvements relocate tirés au hasard depuis S₀) avec T = T₀. Un essai s'arrête après Q itérations sans amélioration ou à la fin de sa part de temps, et on garde la meilleure solution des P essais. Le budget total reste de 180 secondes. `comparer_P.py` compare cette version à notre recuit sur N50, N100 et N200 avec P = 1, 3 et 5.

In [ ]:
%%writefile code/recuit_P.py
# Complément : variante du recuit en P essais indépendants, comme dans le schéma du cours.
# On réutilise le recuit de recuit.py, avec ses options duree_cycle et Q.

import random
import time

from recuit import *

# Schéma du cours : pour k de 1 à P, x0 = solution aléatoire, T = T0, on refroidit
# jusqu'à Q itérations sans amélioration, et on garde la meilleure solution.
# Différence avec notre réchauffage : on ne repart pas de la meilleure solution
# mais d'une nouvelle solution aléatoire (on diversifie au lieu d'intensifier).

def solution_aleatoire(S0, N, nb_pas=None):
    # Solution "aléatoire" mais toujours admissible : on part de S0 et on fait
    # nb_pas mouvements relocate tirés au hasard, en les acceptant tous même
    # s'ils dégradent. Par défaut nb_pas = n, donc chaque mission bouge à peu près une fois.
    if nb_pas is None:
        nb_pas = len(N["interventions"])
    S = {cle: S0[cle][:] for cle in S0}
    pas = essais = 0
    while pas < nb_pas and essais < 100 * nb_pas:
        essais += 1
        V, mouvement = voisin_aleatoire(S, N)
        if V is not None:                     # voisin admissible : on le prend sans condition
            S = V
            pas += 1
    return S


def recuit_essais_P(S0, N, temps_max=180, P=3, Q=None, graine=None, tau0=0.1, nb_pas=None):
    # Recuit comme dans le cours : P essais indépendants, chacun depuis une solution aléatoire.
    # - le budget total est partagé : chaque essai reçoit (temps restant / essais restants),
    #   donc si un essai s'arrête tôt, le temps qu'il laisse revient aux suivants ;
    # - chaque essai refroidit selon le temps (comme pour l'epsilon-contrainte) pour faire
    #   un cycle complet de T0 à T0/1000 dans sa part de temps ;
    # - un essai s'arrête après Q itérations sans amélioration. Par défaut Q = 10 000 x n,
    #   environ 2 fois le plus long écart entre deux améliorations qu'on a mesuré
    #   (à peu près 5 000 x n sur N50), pour ne couper un essai que quand il est vraiment figé.
    random.seed(graine)
    debut = time.perf_counter()
    n = len(N["interventions"])
    if Q is None:
        Q = 10000 * n
    T0 = calculer_T0(S0, N, tau0)              # même T0 pour tous les essais

    meilleure, meilleur_cout, essais = None, float("inf"), []
    for k in range(P):
        reste = temps_max - (time.perf_counter() - debut)
        if reste <= 0:
            break
        part = reste / (P - k)
        x0 = solution_aleatoire(S0, N, nb_pas)
        g = None if graine is None else graine * 1000 + k
        S, c, infos = recuit_simule(x0, N, temps_max=part, graine=g, T0=T0,
                                    duree_cycle=0.97 * part, Q=Q)
        # on garde le détail de chaque essai pour l'analyse
        essais.append({"essai": k + 1, "cout_depart": round(f(x0, N), 2), "cout": round(c, 2),
                       "temps": round(infos["temps"], 2), "iterations": infos["iterations"]})
        if c < meilleur_cout:
            meilleure, meilleur_cout = S, c

    infos = {"essais": essais, "T0": T0, "Q": Q, "temps": time.perf_counter() - debut}
    return meilleure, meilleur_cout, infos

In [ ]:
%%writefile code/comparer_P.py
# Comparaison ajoutée après la soutenance : notre recuit (réchauffage depuis la meilleure
# solution) contre la version du cours (P essais indépendants depuis une solution aléatoire).
# Même budget (180 s), mêmes instances, mêmes graines pour toutes les variantes.
# À lancer depuis le dossier du notebook : python code/comparer_P.py
# Résultats : resultats/variante_P/comparaison_P.json

import json, os, sys, time
from multiprocessing import Pool
ICI = os.path.dirname(os.path.abspath(__file__))
RACINE = os.path.dirname(ICI)
sys.path.insert(0, ICI)
from recuit_P import *

BUDGET = 180
INSTANCES = ["N50", "N100", "N200"]
GRAINES = [1, 2, 3, 4, 5]
# "actuel" = notre recuit ; P1, P3, P5 = version du cours avec 1, 3 ou 5 essais
VARIANTES = ["actuel", "P1", "P3", "P5"]
SORTIE = os.path.join(RACINE, "resultats", "variante_P", "comparaison_P.json")


def charger(nom):
    N = lire_instance(os.path.join(RACINE, "donnees", nom))
    preparer_instance(N)
    return N, json.load(open(os.path.join(RACINE, "S0_communs.json")))[nom]


def lancer(tache):
    # un essai = une instance, une variante, une graine
    nom, variante, graine = tache
    N, S0 = charger(nom)
    if variante == "actuel":
        S, c, inf = recuit_simule(S0, N, temps_max=BUDGET, graine=graine)
        detail = {"rechauffages": inf["rechauffages"], "iterations": inf["iterations"]}
    else:
        # "P3" -> P = 3
        S, c, inf = recuit_essais_P(S0, N, temps_max=BUDGET, P=int(variante[1:]), graine=graine)
        detail = {"essais": inf["essais"]}
    # on vérifie quand même que la solution est bonne et que le coût est juste
    assert est_realisable(S, N) and abs(f(S, N) - c) < 1e-6
    return {"instance": nom, "variante": variante, "graine": graine, "S0": round(f(S0, N), 2),
            "cout": round(c, 2), **detail}


if __name__ == "__main__":
    taches = [(nom, v, g) for g in GRAINES for nom in INSTANCES for v in VARIANTES]
    resultats = []
    debut = time.time()
    os.makedirs(os.path.dirname(SORTIE), exist_ok=True)
    with Pool(4) as p:
        for r in p.imap_unordered(lancer, taches):
            resultats.append(r)
            # on réécrit le fichier à chaque essai, comme ça rien n'est perdu si on arrête avant la fin
            json.dump(resultats, open(SORTIE, "w"), indent=1)
            print(f"[{len(resultats)}/{len(taches)}] {time.time()-debut:.0f}s", r["instance"], r["variante"], r["graine"], r["cout"], flush=True)

### 7.2 Algorithme génétique : temps long et renouvellement

- `genetique_renouvellement.py` : le même génétique, dont les deux taux d'immigration (habituel et après stagnation) deviennent des paramètres ;
- `experiences.py` : prolonge le génétique jusqu'à 400 secondes sur N200 (avec cinq recuits de 180 secondes comme référence), et compare trois politiques d'immigration sur N50 (0/0, 10/20 et 20/40 %) ;
- `long_terme_N200.py` et `renouvellement_N50.py` : lancent chacune des deux études séparément ;
- `analyser_apports.py` : tableaux et figures des deux études.

In [ ]:
%%writefile code/genetique_renouvellement.py
# Partie Axel : algorithme génétique, version pour l'étude du renouvellement (après la soutenance).
# C'est exactement genetique.py, sauf que les deux taux d'immigration (habituel et après
# stagnation) deviennent des paramètres : taux_immigration et taux_stagnation.
# Avec les valeurs par défaut (0,2 et 0,4), on retrouve l'algorithme final.
#
# Algorithme génétique.
# Une solution (un individu) est le planning lui-même : une liste ordonnée de missions
# par technicien, plus la liste des reportées. On réutilise f, est_realisable,
# verifier_tournee et distance_tournee du socle commun.
# Les fonctions finissent par _ag pour ne pas se mélanger avec celles des autres parties.

from socle import *
import random
import time
import pandas as pd


def copie_ag(S):
    # copie d'un planning (on copie chaque liste pour ne pas modifier l'original)
    return {cle: missions[:] for cle, missions in S.items()}


def cle_ag(S, N):
    # "Empreinte" d'un planning, pour repérer les doublons dans la population.
    # L'ordre des reports n'a aucune importance, donc on les trie.
    return (tuple(tuple(S[t]) for t in N["techs"]),
            tuple(sorted(S["reportees"])))


def completer_ag(S, ordre, N, rng, limite, diversifier=False):
    # Essaie d'insérer les missions reportées (dans l'ordre donné) sans rendre le
    # planning impossible. Une mission reste reportée si aucune insertion n'est retenue.
    # limite = heure (perf_counter) à laquelle il faut s'arrêter.
    for mission in ordre:
        if time.perf_counter() >= limite:
            break
        if mission not in S["reportees"]:
            continue

        # on teste toutes les positions chez tous les techniciens
        choix = []
        for tech in N["techs"]:
            ancienne_distance = distance_tournee(N, tech, S[tech])
            for position in range(len(S[tech]) + 1):
                if time.perf_counter() >= limite:
                    return S
                S[tech].insert(position, mission)
                if verifier_tournee(S, N, tech):
                    ajout = distance_tournee(N, tech, S[tech]) - ancienne_distance
                    choix.append((ajout, tech, position))
                S[tech].pop(position)

        if choix:
            choix.sort()
            # Diversification : on choisit au hasard parmi les 3 meilleures insertions.
            # Sinon on prend la meilleure.
            ajout, tech, position = rng.choice(choix[:3]) if diversifier else choix[0]
            # sans diversification, on n'insère que si ça coûte moins cher que le report
            if diversifier or ajout <= N["penalite"]:
                S[tech].insert(position, mission)
                S["reportees"].remove(mission)
    return S


def individu_aleatoire_ag(N, rng, limite):
    # Nouvel individu : même principe que le glouton, mais avec un ordre des missions
    # mélangé et des insertions un peu au hasard, pour avoir des individus différents.
    ordre = list(N["duree"])
    rng.shuffle(ordre)
    S = {tech: [] for tech in N["techs"]}
    S["reportees"] = ordre[:]
    return completer_ag(S, ordre, N, rng, limite, diversifier=True)


def croisement_ag(parent1, parent2, N, rng, limite):
    # L'enfant part de "tout reporté" puis récupère des tournées des parents.
    enfant = {tech: [] for tech in N["techs"]}
    enfant["reportees"] = list(N["duree"])

    # Pour chaque technicien, hériter de la tournée d'un des deux parents.
    # Une mission déjà prise ailleurs n'est pas copiée une seconde fois.
    techniciens = N["techs"][:]
    rng.shuffle(techniciens)
    for tech in techniciens:
        parent = rng.choice([parent1, parent2])
        for mission in parent[tech]:
            if time.perf_counter() >= limite:
                return enfant
            if mission in enfant["reportees"]:
                enfant[tech].append(mission)
                # si la tournée devient impossible, on enlève la mission
                if verifier_tournee(enfant, N, tech):
                    enfant["reportees"].remove(mission)
                else:
                    enfant[tech].pop()

    # Réparation : on essaie de réinsérer les missions encore reportées.
    ordre = enfant["reportees"][:]
    rng.shuffle(ordre)
    return completer_ag(enfant, ordre, N, rng, limite)


def mutation_ag(S, N, rng, limite):
    # Petit changement aléatoire. On fait jusqu'à 20 tentatives et on garde la première
    # qui donne un planning différent et réalisable. Sinon on renvoie une copie de S.
    listes = N["techs"] + ["reportees"]
    for _ in range(20):
        if time.perf_counter() >= limite:
            break
        voisin = copie_ag(S)
        non_vides = [cle for cle in listes if voisin[cle]]
        if not non_vides:
            break
        source = rng.choice(non_vides)
        i = rng.randrange(len(voisin[source]))

        if rng.random() < 0.7:
            # 70 % du temps : déplacer une mission (la réordonner, la reporter ou la réinsérer)
            destination = rng.choice(listes)
            if source == destination == "reportees":
                continue
            mission = voisin[source].pop(i)
            position = rng.randrange(len(voisin[destination]) + 1)
            voisin[destination].insert(position, mission)
        else:
            # 30 % du temps : échanger deux missions ; avec les reports, ça fait un remplacement
            destination = rng.choice(non_vides)
            j = rng.randrange(len(voisin[destination]))
            voisin[source][i], voisin[destination][j] = (
                voisin[destination][j], voisin[source][i])

        if cle_ag(voisin, N) != cle_ag(S, N) and est_realisable(voisin, N):
            return voisin
    return copie_ag(S)


def algorithme_genetique(S0, N, taille_population=20, proba_croisement=0.8,
                        proba_mutation=0.25, budgets=(60, 120, 180),
                        graines=(1, 2, 3, 4, 5), diversite=True,
                        taux_immigration=0.2, taux_stagnation=0.4):
    # Lance une exécution par graine. On relève le meilleur coût à chaque budget
    # (60, 120, 180 s) dans la même exécution.
    # diversite=False ne sert qu'au test d'ablation (population sans immigrants ni refus des doublons).

    # quelques contrôles sur les paramètres
    if taille_population < 4:
        raise ValueError("La population doit contenir au moins 4 individus.")
    if not 0 <= proba_croisement <= 1 or not 0 <= proba_mutation <= 1:
        raise ValueError("Les probabilités doivent être entre 0 et 1.")
    if not 0 <= taux_immigration <= 1 or not 0 <= taux_stagnation <= 1:
        raise ValueError("Les taux d’immigration doivent être compris entre 0 et 1.")
    budgets = sorted(set(budgets))
    if not budgets or budgets[0] <= 0:
        raise ValueError("Les budgets doivent être strictement positifs.")
    if not est_realisable(S0, N):
        raise ValueError("La solution initiale n'est pas réalisable.")

    resultats, solutions = [], []
    for graine in graines:
        rng = random.Random(graine)      # générateur propre à cette graine
        meilleure = copie_ag(S0)
        meilleur_cout = f(S0, N)
        couts = {budget: None for budget in budgets}
        historique = [(0.0, meilleur_cout)]
        generations = stagnation = 0
        suivi_population = []
        dernier_suivi = -1.0
        debut = time.perf_counter()
        limite = debut + budgets[-1]

        def observer(S):
            # Calcule le coût d'un nouvel individu et met à jour le meilleur connu.
            nonlocal meilleure, meilleur_cout
            cout = f(S, N)
            temps = time.perf_counter() - debut
            # Un résultat trouvé après un seuil ne compte pas avant ce seuil :
            # on fige la valeur d'un budget dès qu'on le dépasse.
            for budget in budgets:
                if temps >= budget and couts[budget] is None:
                    couts[budget] = meilleur_cout
            if temps <= budgets[-1] and cout < meilleur_cout - 1e-9:
                meilleure, meilleur_cout = copie_ag(S), cout
                historique.append((temps, cout))
            return cout

        # Population initiale : S0 + des individus aléatoires.
        # Chaque individu est stocké avec son coût.
        population = [(copie_ag(S0), meilleur_cout)]
        vus = {cle_ag(S0, N)}
        for _ in range(10 * taille_population):
            if len(population) >= taille_population or time.perf_counter() >= limite:
                break
            if diversite:
                S = individu_aleatoire_ag(N, rng, limite)
            else:
                # témoin sans diversité : simples mutations de S0
                S = copie_ag(S0)
                for _ in range(rng.randint(1, 3)):
                    S = mutation_ag(S, N, rng, limite)
            cout = observer(S)
            cle = cle_ag(S, N)
            if not diversite or cle not in vus:
                population.append((S, cout))
                vus.add(cle)

        def selection():
            # Tournoi de 2 : on tire deux individus et on garde le meilleur.
            # Ça garde une pression de sélection modérée.
            candidats = rng.sample(population, min(2, len(population)))
            return min(candidats, key=lambda individu: individu[1])[0]

        # Boucle des générations, jusqu'à la fin du budget
        while time.perf_counter() < limite:
            ancien_record = meilleur_cout
            population.sort(key=lambda individu: individu[1])
            nouvelle = population[:2]  # Élitisme : on garde les deux meilleurs.
            vus = {cle_ag(S, N) for S, cout in nouvelle}

            # Après 10 générations sans progrès, on renforce la diversification :
            # mutation à 0,7 au moins et taux_stagnation d'immigrants au lieu de taux_immigration.
            mutation = max(proba_mutation, 0.7) if diversite and stagnation >= 10 else proba_mutation
            proportion = taux_stagnation if stagnation >= 10 else taux_immigration
            # avec un taux nul, aucun immigrant (politique 0 % / 0 %)
            nb_immigrants = max(1, int(taille_population * proportion)) if diversite and proportion > 0 else 0

            for _ in range(10 * taille_population):
                if len(nouvelle) >= taille_population or time.perf_counter() >= limite:
                    break
                if len(nouvelle) < 2 + nb_immigrants:
                    # immigrant : individu tout neuf
                    enfant = individu_aleatoire_ag(N, rng, limite)
                else:
                    parent1, parent2 = selection(), selection()
                    if rng.random() < proba_croisement:
                        enfant = croisement_ag(parent1, parent2, N, rng, limite)
                    else:
                        enfant = copie_ag(parent1)
                    if rng.random() < mutation:
                        for _ in range(rng.randint(1, 3)):
                            enfant = mutation_ag(enfant, N, rng, limite)

                cout = observer(enfant)
                cle = cle_ag(enfant, N)
                if not diversite or cle not in vus:  # Refuser les doublons dans la population.
                    nouvelle.append((enfant, cout))
                    vus.add(cle)

            population = nouvelle
            generations += 1

            # Suivi de la population environ toutes les 0,25 s (pour les graphiques de diversité)
            temps_suivi = time.perf_counter() - debut
            if temps_suivi - dernier_suivi >= 0.25 and temps_suivi <= budgets[-1]:
                cout_population = sum(c for _, c in population) / len(population)
                uniques = len({cle_ag(s, N) for s, _ in population}) / len(population)

                # Diversité des affectations et des arcs des tournées, par rapport au meilleur membre.
                def arcs(solution):
                    # ensemble des trajets (technicien, de, vers) + des reports
                    ensemble = set()
                    for t in N["techs"]:
                        route = [N["domicile"][t]] + solution[t] + [N["domicile"][t]]
                        ensemble.update((t, a, b) for a, b in zip(route, route[1:]))
                    ensemble.update(("report", m) for m in solution["reportees"])
                    return ensemble
                reference = arcs(min(population, key=lambda x: x[1])[0])
                ecarts = []
                for solution, _ in population:
                    a = arcs(solution)
                    # distance de Jaccard : 1 - (arcs communs / arcs au total)
                    ecarts.append(1 - len(a & reference) / max(1, len(a | reference)))
                suivi_population.append({"temps": temps_suivi, "generation": generations,
                    "cout_moyen": cout_population, "cout_min": min(c for _, c in population),
                    "meilleur": meilleur_cout, "fraction_unique": uniques,
                    "diversite_arcs": sum(ecarts) / len(ecarts), "taille": len(population)})
                dernier_suivi = temps_suivi

            # compteur de stagnation : remis à 0 dès que le meilleur s'améliore
            stagnation = 0 if meilleur_cout < ancien_record - 1e-9 else stagnation + 1

        # fin de l'exécution : on complète les budgets pas encore atteints
        temps_reel = time.perf_counter() - debut
        for budget in budgets:
            if couts[budget] is None:
                couts[budget] = meilleur_cout
        assert est_realisable(meilleure, N)
        resultats.append({
            "Essai": graine,
            **{f"Coût {b}": couts[b] for b in budgets},
            "Reports": len(meilleure["reportees"]),
            "Temps": temps_reel,
            "Générations": generations,
            "Population finale": len(population),
            "Historique": historique,
            "Suivi population": suivi_population
        })
        solutions.append(meilleure)
    return resultats, solutions

In [ ]:
%%writefile code/experiences.py
"""Deux apports séparés : AG original N200 à 400 s ; immigration sur N50 à 20 s.
Aucun calcul au chargement. Les sorties existantes ne sont jamais écrasées.
"""
# Expériences ajoutées après la soutenance pour le génétique :
#  - apport 1 : le génétique final sur N200 prolongé jusqu'à 400 s (+ 5 recuits de 180 s
#    sur la même machine, comme témoin) ;
#  - apport 2 : trois politiques d'immigration sur N50 (0/0, 10/20 et 20/40 %), 20 s.
# Lancement complet (depuis le dossier du notebook) : python code/experiences.py --sortie resultats_apports
# Les scripts long_terme_N200.py et renouvellement_N50.py lancent chaque apport séparément.

import os
# un seul fil de calcul pour les bibliothèques numériques
for name in ['OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'VECLIB_MAXIMUM_THREADS', 'MKL_NUM_THREADS', 'NUMEXPR_NUM_THREADS']:
    os.environ[name] = '1'
import argparse, json, time, datetime, sys, platform, hashlib, multiprocessing, zipfile, traceback, bisect
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed
from socle import lire_instance, preparer_instance, est_realisable, f, heuristique_gloutonne
ROOT = Path(__file__).resolve().parents[1]
# politiques testées : (nom, immigration habituelle, immigration après stagnation)
POLITIQUES = [('0_0', 0., 0.), ('10_20', .1, .2), ('20_40', .2, .4)]
BARRIER = None


def initialiser(barrier):
    # la barrière sert à démarrer les 3 essais N50 d'une même graine en même temps
    global BARRIER
    BARRIER = barrier


def sauvegarder(p, obj):
    # écrit d'abord dans un fichier .tmp puis le renomme : pas de fichier à moitié écrit
    p.parent.mkdir(parents=True, exist_ok=True)
    tmp = p.with_suffix('.tmp')
    tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=2))
    tmp.replace(p)


def modele(nom):
    # données de l'instance + solution de départ propre à chaque apport
    N = lire_instance(str(ROOT / 'donnees' / nom))
    preparer_instance(N)
    if nom == 'N200':
        # N200 : le S0 commun du groupe, le même que dans la campagne principale
        S = json.loads((ROOT / 'S0_communs.json').read_text())['N200']
    else:
        # N50 : départ historique figé, indépendant des départages de pandas.
        S = json.loads((ROOT / 'S0_exploratoires.json').read_text())[nom]
    assert est_realisable(S, N)
    return N, S


def essai(tache):
    # Un essai : selon la phase, génétique long sur N200, recuit témoin sur N200,
    # ou génétique avec une politique d'immigration sur N50.
    phase, configuration, seed, budget, base, renfort, sortie = tache
    nom = 'N50' if phase == 'renouvellement' else 'N200'
    N, S0 = modele(nom)
    avant = json.dumps(S0, sort_keys=True)
    if BARRIER is not None:
        BARRIER.wait(timeout=120)
    date = datetime.datetime.now(datetime.timezone.utc).isoformat()
    debut = time.perf_counter()
    cpu = time.process_time()

    if phase == 'long_terme':
        # Import du fichier final inchangé. Aucun paramètre de renouvellement ajouté.
        from genetique import algorithme_genetique
        seuils = [60, 120, 180, 300, 400]
        rr, ss = algorithme_genetique(S0, N, budgets=tuple(seuils), graines=(seed,))
        r = rr[0]
        S = ss[0]
        couts = {str(t): r[f'Coût {t}'] for t in seuils}
    elif phase == 'recuit':
        # le recuit de la partie 2 (code/recuit.py) avec le réglage du rapport,
        # pour situer le génétique sur la même machine
        from recuit import recuit_simule
        S, c, info = recuit_simule(S0, N, temps_max=180, graine=seed, tau0=.1, alpha=.99, k_palier=20)
        r = {'cout': c, 'infos': info}
        seuils = [60, 120, 180]
        trace = info['historique']
        ts = [x[0] for x in trace]
        # meilleur coût connu à 60, 120 et 180 s, lu dans l'historique
        couts = {str(t): trace[max(0, bisect.bisect_right(ts, t) - 1)][1] for t in seuils}
    else:
        from genetique_renouvellement import algorithme_genetique
        seuils = [20]
        rr, ss = algorithme_genetique(S0, N, budgets=(20,), graines=(seed,), taux_immigration=base, taux_stagnation=renfort)
        r = rr[0]
        S = ss[0]
        couts = {'20': r['Coût 20']}

    cpu = time.process_time() - cpu
    duree = time.perf_counter() - debut
    # S0 ne doit pas avoir été modifié, et la solution finale doit être réalisable
    assert json.dumps(S0, sort_keys=True) == avant and est_realisable(S, N)
    obj = dict(phase=phase, configuration=configuration, instance=nom, graine=seed, budget_s=budget,
               solution_initiale=S0, reference=f(S0, N), resultat=r, solution_finale=S, couts_seuils=couts,
               cout_final=f(S, N), reports_finaux=len(S['reportees']), distance_finale=f(S, N) - N['penalite'] * len(S['reportees']),
               temps_mural_s=duree, temps_cpu_s=cpu, date_debut_utc=date,
               parametres={'population': 20, 'croisement': .8, 'mutation_base': .25, 'elites': 2, 'seuil_stagnation_generations': 10, 'mutation_stagnation_min': .7, 'immigration_habituelle': .2 if phase == 'long_terme' else base, 'immigration_stagnation': .4 if phase == 'long_terme' else renfort} if phase != 'recuit' else {'tau0': .1, 'alpha': .99, 'k_palier': 20},
               code='genetique.py final inchangé' if phase == 'long_terme' else 'recuit.py final inchangé' if phase == 'recuit' else 'genetique_renouvellement.py ; seuls les taux deviennent paramétrables')
    destination = Path(sortie) / f'{configuration}_graine_{seed}.json'
    sauvegarder(destination, obj)
    return phase, configuration, seed, couts, duree, cpu


def collecter(futures, out, compte, total, debut):
    # affiche l'avancement à chaque essai terminé (et l'écrit dans AVANCEMENT.txt)
    for fut in as_completed(futures):
        phase, c, g, cc, t, cpu = fut.result()
        compte += 1
        msg = f'{compte}/{total} : {phase}, {c}, graine {g}, coûts {cc}, CPU/mural {cpu/t:.3f}. Durée totale {(time.perf_counter()-debut)/60:.1f} min.'
        print(msg, flush=True)
        (ROOT / 'AVANCEMENT.txt').write_text(msg + '\n')
    return compte


def main():
    p = argparse.ArgumentParser(description=__doc__)
    p.add_argument('--sortie', default='resultats')
    a = p.parse_args()
    out = ROOT / a.sortie
    # on refuse d'écrire dans un dossier qui contient déjà des résultats
    if list(out.glob('*/*_graine_*.json')):
        raise RuntimeError('Des sorties existent. Choisir un nouveau dossier --sortie.')
    out.mkdir(parents=True, exist_ok=True)

    # on note le protocole (machine, versions, empreintes des fichiers) pour pouvoir le retrouver
    empreintes = {str(q.relative_to(ROOT)): hashlib.sha256(q.read_bytes()).hexdigest() for q in list((ROOT / 'code').glob('*.py')) + list((ROOT / 'donnees').rglob('*.csv')) + list(ROOT.glob('S0_*.json'))}
    protocole = dict(debut_utc=datetime.datetime.now(datetime.timezone.utc).isoformat(), python=sys.version, plateforme=platform.platform(), cpu_logiques=os.cpu_count(), empreintes=empreintes,
                     apport_1={'instance': 'N200', 'budget_s': 400, 'seuils_s': [60, 120, 180, 300, 400], 'graines': [1, 2, 3, 4, 5], 'algorithme': 'Code final original inchangé ; immigration 20/40.'},
                     temoin={'instance': 'N200', 'methode': 'Recuit original', 'budget_s': 180, 'graines': [1, 2, 3, 4, 5], 'note': 'Témoin actuel sur la même machine. AG 400 s / recuit 180 s est une comparaison à budgets différents.'},
                     apport_2={'instance': 'N50', 'budget_s': 20, 'graines': [1, 2, 3, 4, 5], 'politiques': [[0, 0], [10, 20], [20, 40]], 'depart': 'Glouton historique recalculé, comme l’étude exploratoire de diversification. Différent du S0 harmonisé de la campagne principale.'},
                     execution='N50 : trois processus synchronisés par graine, cinq cohortes de 20 s. Ensuite N200 : au plus quatre processus, cinq AG 400 s puis cinq recuits 180 s dans la file. Aucun calcul N50 en concurrence avec N200.',
                     limites='Une seule instance par apport, cinq graines ; temps muraux et CPU enregistrés. Les deux expériences ne sont pas fusionnées et leurs coûts ne se comparent pas entre instances.')
    sauvegarder(out / 'protocole.json', protocole)
    debut = time.perf_counter()
    compte = 0
    total = 25

    # Apport 2 (N50) : pour chaque graine, les 3 politiques tournent en même temps
    ctx = multiprocessing.get_context('spawn')
    barrier = ctx.Barrier(3)
    with ProcessPoolExecutor(max_workers=3, mp_context=ctx, initializer=initialiser, initargs=(barrier,)) as pool:
        for g in range(1, 6):
            print(f'N50 : début cohorte graine {g}, trois politiques de 20 s.', flush=True)
            futures = [pool.submit(essai, ('renouvellement', c, g, 20, b, r, str(out / 'renouvellement_N50'))) for c, b, r in POLITIQUES]
            compte = collecter(futures, out, compte, total, debut)

    # Apport 1 (N200) : 5 génétiques de 400 s puis 5 recuits de 180 s
    with ProcessPoolExecutor(max_workers=4, mp_context=ctx) as pool:
        taches = [('long_terme', 'AG_original', g, 400, None, None, str(out / 'long_terme_N200')) for g in range(1, 6)]
        taches += [('recuit', 'RS_180', g, 180, None, None, str(out / 'recuit_N200')) for g in range(1, 6)]
        print('N200 : début des essais longs avec le génétique original et du témoin recuit.', flush=True)
        compte = collecter([pool.submit(essai, t) for t in taches], out, compte, total, debut)

    protocole['fin_utc'] = datetime.datetime.now(datetime.timezone.utc).isoformat()
    protocole['duree_campagne_s'] = time.perf_counter() - debut
    sauvegarder(out / 'protocole.json', protocole)
    print('CAMPAGNE TERMINÉE', flush=True)
    (ROOT / 'AVANCEMENT.txt').write_text(f'Terminé : {compte}/{total}, durée {(time.perf_counter()-debut)/60:.1f} min.\n')


if __name__ == '__main__':
    try:
        main()
    except BaseException:
        # en cas d'erreur, on garde le message dans ERREUR.txt
        (ROOT / 'ERREUR.txt').write_text(traceback.format_exc())
        raise

In [ ]:
%%writefile code/long_terme_N200.py
"""Lance uniquement le génétique FINAL ORIGINAL sur N200 jusqu'à 400 s."""
# Apport 1 seul (depuis le dossier du notebook) : python code/long_terme_N200.py
# 5 graines, budget de 400 s.

import argparse, json, hashlib
from concurrent.futures import ProcessPoolExecutor, as_completed
from experiences import ROOT, essai


def main():
    p = argparse.ArgumentParser(description=__doc__)
    p.add_argument('--graines', type=int, nargs='+', default=[1, 2, 3, 4, 5])
    p.add_argument('--processus', type=int, default=4)
    p.add_argument('--sortie', default='nouveaux_essais_N200')
    a = p.parse_args()
    if a.processus < 1:
        p.error('--processus doit être positif')
    out = ROOT / a.sortie
    folder = out / 'long_terme_N200'
    if list(folder.glob('*.json')):
        raise RuntimeError('Choisir un nouveau dossier --sortie')
    out.mkdir(parents=True, exist_ok=True)
    # on garde une trace du protocole et des empreintes du code utilisé
    meta = {'experience': 'AG original N200, budget 400 s', 'graines': a.graines, 'processus': a.processus, 'empreintes': {str(q.relative_to(ROOT)): hashlib.sha256(q.read_bytes()).hexdigest() for q in (ROOT / 'code').glob('*.py')}}
    (out / 'protocole.json').write_text(json.dumps(meta, ensure_ascii=False, indent=2))
    with ProcessPoolExecutor(max_workers=a.processus) as pool:
        fs = [pool.submit(essai, ('long_terme', 'AG_original', g, 400, None, None, str(folder))) for g in a.graines]
        for f in as_completed(fs):
            print(f.result(), flush=True)


if __name__ == '__main__':
    main()

In [ ]:
%%writefile code/renouvellement_N50.py
"""Lance uniquement les politiques 0/0, 10/20 et 20/40 sur N50, cinq graines, 20 s."""
# Apport 2 seul (depuis le dossier du notebook) : python code/renouvellement_N50.py
# Pour chaque graine, les trois politiques sont lancées ensemble.

import argparse, json, hashlib, multiprocessing
from concurrent.futures import ProcessPoolExecutor, as_completed
from experiences import ROOT, essai, initialiser, POLITIQUES


def main():
    p = argparse.ArgumentParser(description=__doc__)
    p.add_argument('--graines', type=int, nargs='+', default=[1, 2, 3, 4, 5])
    p.add_argument('--sortie', default='nouveaux_essais_N50')
    a = p.parse_args()
    out = ROOT / a.sortie
    folder = out / 'renouvellement_N50'
    if list(folder.glob('*.json')):
        raise RuntimeError('Choisir un nouveau dossier --sortie')
    out.mkdir(parents=True, exist_ok=True)
    meta = {'experience': 'Renouvellement N50, budget 20 s', 'graines': a.graines, 'politiques': [[0, 0], [10, 20], [20, 40]], 'empreintes': {str(q.relative_to(ROOT)): hashlib.sha256(q.read_bytes()).hexdigest() for q in (ROOT / 'code').glob('*.py')}}
    (out / 'protocole.json').write_text(json.dumps(meta, ensure_ascii=False, indent=2))
    # la barrière (3 places) fait démarrer les 3 politiques d'une graine ensemble
    ctx = multiprocessing.get_context('spawn')
    barrier = ctx.Barrier(3)
    with ProcessPoolExecutor(max_workers=3, mp_context=ctx, initializer=initialiser, initargs=(barrier,)) as pool:
        for g in a.graines:
            fs = [pool.submit(essai, ('renouvellement', c, g, 20, b, r, str(folder))) for c, b, r in POLITIQUES]
            for f in as_completed(fs):
                print(f.result(), flush=True)


if __name__ == '__main__':
    main()

In [ ]:
%%writefile code/analyser_apports.py
"""Recrée les statistiques et figures des deux apports, sans nouveaux calculs."""
# (Ce fichier s'appelait analyser.py dans le dossier des apports d'Axel. On l'a renommé
# pour ne pas le confondre avec code/analyser.py, qui analyse la campagne principale.)
# Depuis le dossier du notebook : python code/analyser_apports.py --resultats resultats_apports
# Sorties dans analyses/ : tableaux csv, synthese.json, N200_convergence et N50_renouvellement (png et svg).

import os
from pathlib import Path
ROOT = Path(__file__).resolve().parents[1]
os.environ.setdefault('MPLCONFIGDIR', str(ROOT / '.matplotlib'))
import argparse, json, csv, statistics, bisect


def stats(v):
    # petit résumé statistique d'une liste de coûts
    return dict(n=len(v), meilleur=min(v), moyenne=statistics.mean(v), ecart_type=statistics.stdev(v) if len(v) > 1 else 0, pire=max(v), mediane=statistics.median(v))


def lire(d):
    # tous les json d'un dossier, triés par graine
    return sorted([json.loads(p.read_text()) for p in d.glob('*.json')], key=lambda r: r['graine'])


def ecrire(p, rows):
    if not rows:
        return
    with p.open('w', encoding='utf-8-sig', newline='') as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0]), delimiter=';')
        w.writeheader()
        w.writerows(rows)


def main():
    p = argparse.ArgumentParser(description=__doc__)
    p.add_argument('--partiel', action='store_true')
    p.add_argument('--resultats', default='resultats')
    p.add_argument('--sortie', default='analyses')
    p.add_argument('--sans-graphiques', action='store_true')
    a = p.parse_args()
    src = ROOT / a.resultats
    out = ROOT / a.sortie
    out.mkdir(parents=True, exist_ok=True)
    long = lire(src / 'long_terme_N200')
    rs = lire(src / 'recuit_N200')
    n50 = lire(src / 'renouvellement_N50')
    # --partiel permet d'analyser avant la fin de la campagne
    if not a.partiel:
        assert len(long) == 5 and len(rs) == 5 and len(n50) == 15, 'Attendre les 25 sorties'

    # Apport 2 (N50) : statistiques par politique d'immigration
    groupes = {n: [r for r in n50 if r['configuration'] == n] for n in ['0_0', '10_20', '20_40']}
    n50stats = []
    longstats = []
    ind = []
    compar = []
    for n, rr in groupes.items():
        if rr:
            v = [r['cout_final'] for r in rr]
            n50stats.append(dict(politique=n, **stats(v), generations_moyennes=statistics.mean(r['resultat']['Générations'] for r in rr), population_finale_moyenne=statistics.mean(r['resultat']['Population finale'] for r in rr)))
            for r in rr:
                ind.append(dict(experience='renouvellement_N50', configuration=n, graine=r['graine'], budget_s=20, cout=r['cout_final'], generations=r['resultat']['Générations'], temps_cpu_s=r['temps_cpu_s']))

    # comparaison graine par graine avec la politique d'origine 20/40
    base = {r['graine']: r for r in groupes['20_40']}
    for n in ['0_0', '10_20']:
        rr = [r for r in groupes[n] if r['graine'] in base]
        if rr:
            diffs = [r['cout_final'] - base[r['graine']]['cout_final'] for r in rr]
            compar.append(dict(politique=n, reference='20_40', n=len(diffs), ecart_moyen_cout=statistics.mean(diffs), meilleur_sur=sum(x < -1e-7 for x in diffs), egalite_sur=sum(abs(x) <= 1e-7 for x in diffs), moins_bon_sur=sum(x > 1e-7 for x in diffs)))

    # Apport 1 (N200) : coûts du génétique à chaque seuil de temps, et du recuit témoin
    for t in [60, 120, 180, 300, 400]:
        if long:
            longstats.append(dict(methode='AG_original', budget_s=t, **stats([r['couts_seuils'][str(t)] for r in long])))
    for t in [60, 120, 180]:
        if rs:
            longstats.append(dict(methode='Recuit', budget_s=t, **stats([r['couts_seuils'][str(t)] for r in rs])))
    for nom, rr in [('long_terme_N200', long), ('recuit_N200', rs)]:
        for r in rr:
            for t, c in r['couts_seuils'].items():
                ind.append(dict(experience=nom, configuration=r['configuration'], graine=r['graine'], budget_s=int(t), cout=c, generations=r['resultat'].get('Générations', ''), temps_cpu_s=r['temps_cpu_s']))
    for name, rows in [('N50_renouvellement', n50stats), ('N200_long_terme', longstats), ('essais_individuels', ind), ('N50_comparaison_graines', compar)]:
        ecrire(out / f'{name}.csv', rows)
    obj = dict(effectifs={'AG_N200': len(long), 'Recuit_N200': len(rs), 'renouvellement_N50': len(n50)}, N50=n50stats, N200=longstats, comparaisons_N50=compar)
    (out / 'synthese.json').write_text(json.dumps(obj, ensure_ascii=False, indent=2))
    for r in n50stats:
        print('N50', r['politique'], f"{r['moyenne']:.2f} ± {r['ecart_type']:.2f}", flush=True)

    # Graphiques
    if not a.sans_graphiques:
        import numpy as np
        import matplotlib
        matplotlib.use('Agg')
        import matplotlib.pyplot as plt
        plt.rcParams.update({'font.family': 'Arial', 'font.size': 11, 'axes.titlesize': 13, 'axes.titleweight': 'bold', 'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True, 'grid.alpha': .2, 'svg.fonttype': 'none'})

        def save(fig, n):
            for ext in ['png', 'svg']:
                fig.savefig(out / f'{n}.{ext}', dpi=220, facecolor='white')
            plt.close(fig)

        if long:
            # coût moyen du génétique seconde par seconde (+- un écart-type) jusqu'à 400 s
            grid = np.arange(0, 401)
            Y = []
            for r in long:
                tr = r['resultat']['Historique']
                ts = [x[0] for x in tr]
                Y.append([tr[max(0, bisect.bisect_right(ts, float(t)) - 1)][1] for t in grid])
            Y = np.array(Y)
            m = Y.mean(axis=0)
            sd = Y.std(axis=0, ddof=1) if len(Y) > 1 else np.zeros_like(m)
            fig, ax = plt.subplots(figsize=(9, 3.6), layout='constrained')
            ax.step(grid, m, where='post', color='#28647D', lw=2, label='Génétique original : moyenne')
            ax.fill_between(grid, m - sd, m + sd, step='post', color='#28647D', alpha=.15, label='± un écart-type')
            ax.axvline(180, ls=':', color='#777777', lw=1)
            if rs:
                # ligne horizontale : coût moyen du recuit à 180 s
                ref = statistics.mean(r['couts_seuils']['180'] for r in rs)
                ax.plot([180, 400], [ref, ref], ls='--', color='#C57230', lw=1.5, label='Recuit : moyenne à 180 s')
            ax.set(xlim=(0, 400), xlabel='Temps de calcul (s)', ylabel='Meilleur coût pénalisé connu', title='N200 — prolongement du génétique jusqu’à 400 secondes')
            ax.set_xticks([0, 60, 120, 180, 300, 400])
            ax.legend(frameon=False, fontsize=9)
            save(fig, 'N200_convergence')

        if n50:
            # les 5 essais de chaque politique (points) avec moyenne et écart-type
            fig, ax = plt.subplots(figsize=(8.5, 3.5), layout='constrained')
            colors = ['#73828B', '#2E8594', '#C57230']
            for i, (n, col) in enumerate(zip(['0_0', '10_20', '20_40'], colors)):
                rr = groupes[n]
                if not rr:
                    continue
                v = [r['cout_final'] for r in rr]
                # petit décalage horizontal pour que les points ne se cachent pas
                xs = [i + (k - (len(v) - 1) / 2) * .045 for k in range(len(v))]
                ax.scatter(xs, v, color=col, s=38, alpha=.75, zorder=3)
                ax.errorbar(i, statistics.mean(v), yerr=statistics.stdev(v) if len(v) > 1 else 0, fmt='D', ms=7, lw=2, capsize=6, color=col, mec='white', zorder=4)
            ax.set_xticks([0, 1, 2], ['0 % / 0 %', '10 % / 20 %', '20 % / 40 %'])
            ax.set(xlabel='Immigration habituelle / après stagnation', ylabel='Coût final à 20 s', title='N50 — comparaison des trois politiques de renouvellement')
            fig.supxlabel('Points : cinq essais ; losange : moyenne ; barres : ± un écart-type', fontsize=9)
            save(fig, 'N50_renouvellement')
    print('Analyses enregistrées dans', out)


if __name__ == '__main__':
    main()